---

## Notebook Documentation

### Setup & Authentication
- Google Colab Enterprise authentication with service account
- MIMIC-IV 3.1 on BigQuery + local DuckDB (`mimic_iv_derived.duckdb`) on Google Drive
- BigQuery used for: cohort, EDA tables (diagnoses, procedures, services)
- DuckDB used for: all derived features (scores, vitals, labs, temporal features)

---

### Section 0 — Setup
| Step | Cell | Description |
|------|------|-------------|
| 0.0 | 1 | Google Colab authentication & project config |
| 0.1 | 2 | Import libraries |
| 0.1b | 3 | Mount Google Drive, connect DuckDB |
| 0.2 | 4 | Initialize BigQuery client |

---

### Section 1 — Data Extraction
| Step | Cell | Description |
|------|------|-------------|
| 1.0 | 5 | Cohort extraction from BigQuery (adults ≥18, hospital stay ≥72h, first ICU stay only) |
| 1.1 | 6 | Create binary target: ICU LOS ≥21 days = 1 |
| 1.2 | 7 | Static derived features via DuckDB (Charlson, SAPSII, APSIII, OASIS, LODS, SOFA, anthropometrics, sepsis3, AKI) |
| 1.3 | 8 | Temporal features via DuckDB — 3 × 24h bins (0–24h, 24–48h, 48–72h): vitals, GCS, labs, coagulation, blood gas, cardiac markers, enzymes, ventilation, RRT |
| 1.4 | 9 | Merge all features onto cohort_df; procedures query via BigQuery (hadm_id join); export cohort snapshot CSV to Drive |

---

### Section 2 — EDA
| Step | Cell | Description |
|------|------|-------------|
| 2.1 | 10 | Create 4 stratified cohorts: 21+/Died, 21+/Survived, <21/Died, <21/Survived |
| 2.2 | 11 | Insurance type & mortality analysis — 4-group comparison |
| 2.3 | 12 | Diagnosis & LOS analysis — top 15 diagnoses by volume (joins via icustays on hadm_id) |
| 2.4 | 13 | Procedure utilization — 4-group comparison |
| 2.5 | 14 | Readmission & ICU fragmentation analysis |

---

### Section 3 — Feature Engineering
| Step | Cell | Description |
|------|------|-------------|
| 3.1 | 15 | Static feature matrix (demographics, admission type, comorbidities, severity scores) |
| 3.2 | 16 | Temporal feature matrix + delta features (bin-to-bin changes) |
| 3.3 | 17 | StandardScaler preprocessing; combined feature matrix |
| 3.4 | 18 | 3-way stratified split: 20% final holdout / 64% train / 16% validation |

---

### Section 4 — Predictive Modeling
| Step | Cell | Description |
|------|------|-------------|
| 4.1 | 19 | Define evaluation metrics: AUPRC (primary), AUROC, Brier score, Accuracy, Sensitivity, Specificity |
| 4.2 | 20 | Lasso — 5-fold CV alpha selection (LassoCV) |
| 4.3 | 21 | Ridge — 5-fold CV alpha selection (RidgeCV) |
| 4.4 | 22 | Random Forest — 5-fold CV + final fit (200 trees, class_weight balanced) |
| 4.5 | 23 | XGBoost — 5-fold CV + final fit (200 rounds, scale_pos_weight) |
| 4.6 | 24 | Model comparison table (AUPRC, AUROC, Brier, Accuracy, Sensitivity, Specificity) |
| 4.7 | 25 | 4-panel model comparison visualization |
| 4.7b | 26 | Confusion matrices — validation set (all 4 models) |
| 4.7c | 27 | Calibration plots — validation set (reliability curves + Brier + probability histograms) |
| 4.8 | 28 | Precision-Recall and ROC curves — validation set |
| 4.9 | 29 | Feature importance comparison — RF vs XGBoost (top 15) |

---

### Section 5 — Summary & Export
| Step | Cell | Description |
|------|------|-------------|
| 5.1 | 30 | Final cohort summary and best model report |
| 5.2 | 31 | Holdout evaluation (20% never-seen data): all 4 models, PR/ROC curves, confusion matrices, calibration plots |
| 5.3 | 32 | Save & download: models (pkl/json) + CSVs + plots → `icu_outputs.zip` |

---

### Key Design Decisions
- **DuckDB for derived features** — avoids BigQuery quota limits and chunked-query failures
- **3-way split** — 20% holdout is never touched until final evaluation
- **5-fold stratified CV** — on training set only for RF and XGBoost; built into LassoCV/RidgeCV
- **AUPRC as primary metric** — appropriate for imbalanced classes (~2.3% positive)
- **Brier score** — measures probability calibration quality (lower = better)
- **Lasso/Ridge probabilities clipped to [0,1]** — linear models can predict outside this range

---

### Expected Outputs

**Figures** (saved locally + in Drive):
- `05_model_comparison.png`
- `confusion_matrices_val.png`
- `calibration_plots_val.png`
- `06_pr_roc_curves.png`
- `07_feature_importance.png`
- `holdout_curves.png` (Drive)
- `confusion_matrices_holdout.png` (Drive)
- `calibration_plots_holdout.png` (Drive)

**Data files** (in `icu_outputs.zip`):
- `model_lasso.pkl`, `model_ridge.pkl`, `model_rf.pkl` — sklearn models
- `model_xgb.json` — XGBoost native format
- `cohort_df.csv` — full merged feature dataframe
- `validation_metrics.csv`, `holdout_metrics.csv` — results tables

---


# ICU ≥21 Days Comprehensive Analysis & Predictive Modeling
## MIMIC-IV 3.1 Deep Dive: Clinical Trajectories, Procedures, and Risk Stratification

### Notebook Objectives
This notebook provides a **production-grade analysis** of ICU patients categorized by length of stay (LOS):

1. **Exploratory Data Analysis (EDA)** - Compare 21+ vs <21 day stays
2. **Procedure Analysis** - Procedure balance and utilization patterns
3. **Predictive Modeling** - 5 models (Lasso, Ridge, RF, XGBoost, TabPFN)
4. **Output** - Publication-quality figures and model comparison


# STEP 0.0: AUTHENTICATION & PROJECT SETUP
# Transplanted from WORKING notebook — this exact sequence is confirmed to work.

In [ ]:


from google.colab import auth
import os

auth.authenticate_user()

# Explicitly set credentials (same as working notebook)
from google.auth import default
credentials, project = default()
os.environ['GOOGLE_CLOUD_PROJECT'] = project

print(f"Authenticated as: {credentials.service_account_email if hasattr(credentials, 'service_account_email') else 'User'}")
print(f"Project: {project}")

# Project / dataset config
PROJECT_ID         = "ml-mps-trl-fldt01-p-0541"
DATASET_PROJECT_ID = 'physionet-data'
DATASET_ID         = 'mimiciv_3_1_hosp'
LOCATION           = 'US'

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

print(f"Dataset:  {DATASET_PROJECT_ID}.{DATASET_ID}")
print("✓ Configuration complete")


# STEP 0.1: IMPORT LIBRARIES

In [ ]:

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

from sklearn.linear_model import Lasso, Ridge, LassoCV, RidgeCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score, auc, precision_recall_curve, confusion_matrix,
    classification_report, brier_score_loss, roc_curve, accuracy_score
)

from google.cloud import bigquery

NP_SEED = 42
np.random.seed(NP_SEED)
print("✓ All libraries imported successfully")

# STEP 0.1b: GOOGLE DRIVE MOUNT & MIMIC-IV DERIVED FEATURES (DuckDB)

In [ ]:

# ──────────────────────────────────────────────────────────────────────────────
# Mounts your Drive and connects to mimic_iv_derived.duckdb.
# Prints every table/view with column names and dtypes so you know
# what derived features are available before wiring them into models.
# ──────────────────────────────────────────────────────────────────────────────

# Install duckdb (pre-installed in newer Colab images, but this is safe)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'], check=True)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import duckdb
import pandas as pd

# ── Adjust this path if the file lives in a sub-folder ───────────────────────
DUCKDB_PATH = '/content/drive/MyDrive/mimic_iv_derived.duckdb'

try:
    duck = duckdb.connect(DUCKDB_PATH, read_only=True)
    print(f"✓ Connected to DuckDB: {DUCKDB_PATH}\n")

    # List all tables and views
    catalog = duck.execute(
        "SELECT table_name, table_type "
        "FROM information_schema.tables "
        "ORDER BY table_type, table_name"
    ).fetchdf()

    if catalog.empty:
        print("⚠  No tables found in this database.")
    else:
        print(f"{'='*70}")
        print(f"  DERIVED TABLES / VIEWS  ({len(catalog)} total)")
        print(f"{'='*70}")

        for _, row in catalog.iterrows():
            tname = row['table_name']
            ttype = row['table_type']

            # Row count
            try:
                nrows = duck.execute(f'SELECT COUNT(*) FROM "{tname}"').fetchone()[0]
            except Exception:
                nrows = '?'

            # Column schema
            schema = duck.execute(f'DESCRIBE "{tname}"').fetchdf()

            print(f"\n  [{ttype}]  {tname}  ({nrows:,} rows)")
            print(f"  {'-'*60}")
            for _, col in schema.iterrows():
                col_name  = col.get('column_name', col.iloc[0])
                col_type  = col.get('column_type', col.iloc[1])
                nullable  = col.get('null', col.iloc[2]) if len(col) > 2 else ''
                print(f"    {col_name:<35} {col_type:<20} {nullable}")

        print(f"\n{'='*70}")
        print("  ✓ Schema inspection complete.")
        print("  Use duck.execute('SELECT * FROM <table_name> LIMIT 5').fetchdf()")
        print("  to preview any table before joining to cohort_df.")
        print(f"{'='*70}")

except FileNotFoundError:
    print(f"\n✗ File not found: {DUCKDB_PATH}")
    print("  Check that the file name and Drive path are correct.")
    print("  If it's in a subfolder, update DUCKDB_PATH above.")
except Exception as e:
    print(f"\n✗ DuckDB connection error: {e}")


# STEP 0.2: INITIALIZE BIGQUERY CLIENT
# Transplanted from WORKING notebook — confirmed working configuration.

In [ ]:

from google.cloud import bigquery

def_config = bigquery.job.QueryJobConfig(
    default_dataset=DATASET_PROJECT_ID + "." + DATASET_ID
)

client = bigquery.Client(
    project=PROJECT_ID,
    location='US',
    default_query_job_config=def_config
)

print(f"✓ BigQuery client initialized")
print(f"  Project:  {client.project}")
print(f"  Location: {client.location}")

# Test connection
test_query = 'SELECT COUNT(*) as cnt FROM `physionet-data.mimiciv_3_1_hosp.patients` LIMIT 1'
try:
    result = client.query(test_query).to_dataframe()
    print(f"✓ MIMIC-IV 3.1 accessible — patient table reachable")
except Exception as e:
    print(f"ERROR: {e}")


# STEP 1.0: COHORT EXTRACTION
# Filters: adult (≥18), hospital stay ≥72h, exclude deaths before 72h
# (outcome is censored if the patient dies before the 3-day feature window closes)

In [ ]:


cohort_query = """
SELECT
    ie.stay_id, ie.subject_id, ie.hadm_id,
    ie.intime, ie.outtime, ie.first_careunit, ie.last_careunit,
    ROUND(DATETIME_DIFF(ie.outtime, ie.intime, MINUTE)/1440.0, 4) AS los_days,
    p.gender,
    p.anchor_age + (EXTRACT(YEAR FROM ie.intime) - p.anchor_year) AS age_at_icu,
    adm.race, adm.marital_status, adm.language, adm.insurance,
    adm.admission_type, adm.admission_location, adm.discharge_location,
    adm.admittime, adm.dischtime, adm.edregtime,
    adm.hospital_expire_flag,
    ROUND(DATETIME_DIFF(adm.dischtime, adm.admittime, MINUTE)/1440.0, 2) AS los_hospital,
    CASE WHEN adm.edregtime IS NOT NULL
         THEN ROUND(DATETIME_DIFF(adm.admittime, adm.edregtime, MINUTE)/60.0, 2)
         ELSE NULL END AS ed_los_hours,
    ROUND(DATETIME_DIFF(ie.intime, adm.admittime, MINUTE)/60.0, 2) AS icu_admit_delay_hours,
    EXTRACT(HOUR      FROM ie.intime) AS admit_hour,
    EXTRACT(DAYOFWEEK FROM ie.intime) AS admit_dow,
    EXTRACT(MONTH     FROM ie.intime) AS admit_month,
    CASE WHEN EXTRACT(DAYOFWEEK FROM ie.intime) IN (1,7) THEN 1 ELSE 0 END AS weekend_admit,
    CASE WHEN EXTRACT(MONTH FROM ie.intime) IN (12,1,2) THEN 1
         WHEN EXTRACT(MONTH FROM ie.intime) IN (3,4,5)  THEN 2
         WHEN EXTRACT(MONTH FROM ie.intime) IN (6,7,8)  THEN 3
         ELSE 4 END AS admit_season,
    COUNT(DISTINCT dx.icd_code) AS num_diagnoses
FROM `physionet-data.mimiciv_3_1_icu.icustays` ie
JOIN `physionet-data.mimiciv_3_1_hosp.patients`   p   USING (subject_id)
JOIN `physionet-data.mimiciv_3_1_hosp.admissions` adm USING (hadm_id)
LEFT JOIN `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` dx ON dx.hadm_id = ie.hadm_id
WHERE p.anchor_age >= 18
  AND DATETIME_DIFF(adm.dischtime, adm.admittime, HOUR) >= 72
  AND NOT (adm.hospital_expire_flag = 1
           AND DATETIME_DIFF(adm.dischtime, adm.admittime, HOUR) < 72)
GROUP BY
    ie.stay_id, ie.subject_id, ie.hadm_id, ie.intime, ie.outtime,
    ie.first_careunit, ie.last_careunit, los_days, p.gender, age_at_icu,
    adm.race, adm.marital_status, adm.language, adm.insurance,
    adm.admission_type, adm.admission_location, adm.discharge_location,
    adm.admittime, adm.dischtime, adm.edregtime, adm.hospital_expire_flag,
    los_hospital, ed_los_hours, icu_admit_delay_hours,
    admit_hour, admit_dow, admit_month, weekend_admit, admit_season

"""

cohort_df = client.query(cohort_query).to_dataframe()
for col in ['intime','outtime','admittime','dischtime']:
    cohort_df[col] = pd.to_datetime(cohort_df[col])

print(f"Cohort: {len(cohort_df):,} stays | {cohort_df['subject_id'].nunique():,} patients")
print(f"  Age (mean):   {cohort_df['age_at_icu'].mean():.1f}")
print(f"  LOS (median): {cohort_df['los_days'].median():.1f} d")
print(f"  Mortality:    {cohort_df['hospital_expire_flag'].mean():.1%}")
print(f"  ICU units:")
print(cohort_df['first_careunit'].value_counts().to_string())


# STEP 1.2: CREATE TARGET VARIABLE (21+ DAYS)

In [ ]:

print("Creating target variable...\\n")

THRESHOLD_DAYS = 21
cohort_df['icu_stay_21plus'] = (cohort_df['los_days'] >= THRESHOLD_DAYS).astype(int)
cohort_df['los_category'] = pd.cut(
    cohort_df['los_days'],
    bins=[0, 7, 14, 21, 365],
    labels=['<7d', '7-14d', '14-21d', '21+d'],
    include_lowest=True
)

print(f"Target Distribution:")
print(f"  <21 days: {(cohort_df['icu_stay_21plus']==0).sum():,} ({(cohort_df['icu_stay_21plus']==0).mean():.1%})")
print(f"  >=21 days: {(cohort_df['icu_stay_21plus']==1).sum():,} ({(cohort_df['icu_stay_21plus']==1).mean():.1%})")
print(f"  Class Imbalance: {(cohort_df['icu_stay_21plus']==0).sum() / (cohort_df['icu_stay_21plus']==1).sum():.2f}:1")

# STEP 1.2: STATIC DERIVED FEATURES  (all queries via DuckDB)
# Register cohort IDs once so DuckDB can use them in every query below.

In [ ]:


stay_ids = cohort_df['stay_id'].tolist()
hadm_ids = cohort_df['hadm_id'].tolist()

duck.register('_cohort_stays', cohort_df[['stay_id']].drop_duplicates())
duck.register('_cohort_hadms', cohort_df[['hadm_id']].drop_duplicates())

# ── 1. Charlson comorbidity (hadm_id) ─────────────────────────────────────────
print("Charlson comorbidities...")
try:
    charlson_df = duck.execute("""
SELECT c.hadm_id, c.charlson_comorbidity_index,
    c.myocardial_infarct, c.congestive_heart_failure, c.chronic_pulmonary_disease,
    c.renal_disease, c.severe_liver_disease, c.metastatic_solid_tumor,
    c.diabetes_with_cc, c.dementia, c.cerebrovascular_disease
FROM charlson c
INNER JOIN _cohort_hadms h ON c.hadm_id = h.hadm_id
""").fetchdf()
    print(f"  charlson: {len(charlson_df):,} rows")
except Exception as e:
    charlson_df = pd.DataFrame(); print(f"  charlson FAILED: {e}")

# ── 2. Anthropometrics (stay_id) ──────────────────────────────────────────────
print("Anthropometrics...")
try:
    height_df = duck.execute("""
SELECT stay_id, CAST(height AS DOUBLE) AS height
FROM first_day_height
WHERE stay_id IN (SELECT stay_id FROM _cohort_stays)
""").fetchdf()
    weight_df = duck.execute("""
SELECT stay_id, weight
FROM first_day_weight
WHERE stay_id IN (SELECT stay_id FROM _cohort_stays)
""").fetchdf()
    anthro_df = height_df.merge(weight_df, on='stay_id', how='outer')
    anthro_df['bmi'] = np.where(
        (anthro_df['height'] > 0) & (anthro_df['weight'] > 0),
        anthro_df['weight'] / ((anthro_df['height'] / 100) ** 2), np.nan)
    print(f"  anthropometrics: {len(anthro_df):,} rows")
except Exception as e:
    anthro_df = pd.DataFrame(); print(f"  anthropometrics FAILED: {e}")

# ── 3. Severity scores (stay_id) ──────────────────────────────────────────────
print("Severity scores...")
try:
    sapsii_df  = duck.execute("SELECT stay_id, sapsii  AS sapsii_score  FROM sapsii  WHERE stay_id IN (SELECT stay_id FROM _cohort_stays)").fetchdf()
    apsiii_df  = duck.execute("SELECT stay_id, apsiii  AS apsiii_score  FROM apsiii  WHERE stay_id IN (SELECT stay_id FROM _cohort_stays)").fetchdf()
    oasis_df   = duck.execute("SELECT stay_id, oasis   AS oasis_score   FROM oasis   WHERE stay_id IN (SELECT stay_id FROM _cohort_stays)").fetchdf()
    lods_df    = duck.execute("SELECT stay_id, lods    AS lods_score    FROM lods    WHERE stay_id IN (SELECT stay_id FROM _cohort_stays)").fetchdf()
    sofa_fd_df = duck.execute("SELECT stay_id, sofa    AS first_day_sofa FROM first_day_sofa WHERE stay_id IN (SELECT stay_id FROM _cohort_stays)").fetchdf()
    severity_df = cohort_df[['stay_id']].drop_duplicates()
    for df in [sapsii_df, apsiii_df, oasis_df, lods_df, sofa_fd_df]:
        severity_df = severity_df.merge(df, on='stay_id', how='left')
    print(f"  severity scores: {len(severity_df):,} rows")
except Exception as e:
    severity_df = pd.DataFrame({'stay_id': cohort_df['stay_id']}); print(f"  severity FAILED: {e}")

# ── 4. Admitting service (BigQuery — not in DuckDB) ───────────────────────────
print("Services...")
SURGICAL = frozenset(['CSURG','NSURG','ORTHO','SURG','TSURG','VSURG','TRAUM'])

def run_chunked(query_fn, ids, chunk_size=5000):
    frames = []
    for i in range(0, len(ids), chunk_size):
        ids_str = ','.join(map(str, ids[i:i+chunk_size]))
        try:
            frames.append(client.query(query_fn(ids_str)).to_dataframe())
        except Exception as e:
            print(f"  chunk {i//chunk_size+1} error: {e}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

try:
    services_df = run_chunked(lambda ids: f"""
SELECT svc.hadm_id,
    ARRAY_AGG(svc.curr_service ORDER BY svc.transfertime LIMIT 1)[OFFSET(0)] AS admitting_service,
    COUNT(*) - 1 AS n_service_transfers
FROM `physionet-data.mimiciv_3_1_hosp.services` svc
JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie USING (hadm_id)
WHERE ie.stay_id IN ({ids})
  AND svc.transfertime <= DATETIME_ADD(ie.intime, INTERVAL 72 HOUR)
GROUP BY svc.hadm_id
""", stay_ids)
    services_df['surgical_admission'] = services_df['admitting_service'].isin(SURGICAL).astype(int)
    print(f"  services: {len(services_df):,} rows")
except Exception as e:
    services_df = pd.DataFrame(); print(f"  services FAILED: {e}")

# ── 5. Sepsis-3 (stay_id) ─────────────────────────────────────────────────────
print("Sepsis-3...")
try:
    sepsis_df = duck.execute("""
SELECT stay_id, CAST(sepsis3 AS INTEGER) AS sepsis3_flag
FROM sepsis3
WHERE stay_id IN (SELECT stay_id FROM _cohort_stays)
""").fetchdf()
    print(f"  sepsis3: {len(sepsis_df):,} rows")
except Exception as e:
    sepsis_df = pd.DataFrame(); print(f"  sepsis3 FAILED: {e}")

# ── 6. AKI / KDIGO first 72h (stay_id) ───────────────────────────────────────
print("AKI/KDIGO...")
try:
    aki_df = duck.execute("""
SELECT k.stay_id,
    MAX(COALESCE(k.aki_stage_creat, 0)) AS aki_stage_creat_max,
    MAX(COALESCE(k.aki_stage_uo,    0)) AS aki_stage_uo_max,
    MAX(COALESCE(k.aki_stage,       0)) AS aki_max_stage
FROM kdigo_stages k
INNER JOIN icustay_detail ict ON k.stay_id = ict.stay_id
WHERE k.stay_id IN (SELECT stay_id FROM _cohort_stays)
  AND k.charttime >= ict.icu_intime
  AND datediff('hour', ict.icu_intime, k.charttime) <= 72
GROUP BY k.stay_id
""").fetchdf()
    print(f"  aki/kdigo: {len(aki_df):,} rows")
except Exception as e:
    aki_df = pd.DataFrame(); print(f"  aki/kdigo FAILED: {e}")

print("\n✓ Static derived extraction complete")


# STEP 1.3: TEMPORAL FEATURES — 0-24h (bin 0), 24-48h (bin 1), 48-72h (bin 2)
# All queries run against DuckDB using datediff() instead of BigQuery DATETIME_DIFF.

In [ ]:


temporal_frames = {}

def dbin(ct, it):
    """Return a DuckDB CASE expression that assigns 24h bin labels 0/1/2."""
    return (f"CASE WHEN datediff('hour',{it},{ct})<24 THEN 0 "
            f"     WHEN datediff('hour',{it},{ct})<48 THEN 1 "
            f"     ELSE 2 END")

# ── VITALS (stay_id) ──────────────────────────────────────────────────────────
print("Vitals...")
try:
    temporal_frames['vitals'] = duck.execute(f"""
WITH b AS (
    SELECT v.stay_id, {dbin('v.charttime','ict.icu_intime')} AS bin,
        v.heart_rate, v.sbp, v.dbp, v.mbp, v.resp_rate,
        CAST(v.temperature AS DOUBLE) AS temperature, v.spo2
    FROM vitalsign v
    INNER JOIN icustay_detail ict ON v.stay_id = ict.stay_id
    WHERE v.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND v.charttime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, v.charttime) BETWEEN 0 AND 71)
SELECT stay_id, bin,
    ROUND(AVG(heart_rate),2) AS hr_mean,  ROUND(MIN(heart_rate),2) AS hr_min,  ROUND(MAX(heart_rate),2) AS hr_max,
    ROUND(AVG(sbp),2)        AS sbp_mean, ROUND(MIN(sbp),2)        AS sbp_min,
    ROUND(AVG(dbp),2)        AS dbp_mean,
    ROUND(AVG(mbp),2)        AS mbp_mean, ROUND(MIN(mbp),2)        AS mbp_min,
    ROUND(AVG(resp_rate),2)  AS rr_mean,  ROUND(MAX(resp_rate),2)  AS rr_max,
    ROUND(AVG(temperature),2) AS temp_mean,
    ROUND(AVG(spo2),2)       AS spo2_mean, ROUND(MIN(spo2),2)      AS spo2_min
FROM b GROUP BY stay_id, bin
""").fetchdf()
    print(f"  vitals: {len(temporal_frames['vitals']):,} rows")
except Exception as e:
    temporal_frames['vitals'] = pd.DataFrame(); print(f"  vitals FAILED: {e}")

# ── GCS (stay_id) — fixed column names: gcs (not gcs_total), gcs_eyes ─────────
print("GCS...")
try:
    temporal_frames['gcs'] = duck.execute(f"""
WITH b AS (
    SELECT g.stay_id, {dbin('g.charttime','ict.icu_intime')} AS bin,
        g.gcs AS gcs_total
    FROM gcs g
    INNER JOIN icustay_detail ict ON g.stay_id = ict.stay_id
    WHERE g.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND g.charttime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, g.charttime) BETWEEN 0 AND 71)
SELECT stay_id, bin,
    ROUND(AVG(gcs_total),2) AS gcs_mean, ROUND(MIN(gcs_total),2) AS gcs_min
FROM b GROUP BY stay_id, bin
""").fetchdf()
    print(f"  gcs: {len(temporal_frames['gcs']):,} rows")
except Exception as e:
    temporal_frames['gcs'] = pd.DataFrame(); print(f"  gcs FAILED: {e}")

# ── CHEMISTRY (hadm_id→stay_id via icustay_detail) — no magnesium/phosphate ──
print("Chemistry...")
try:
    temporal_frames['chemistry'] = duck.execute(f"""
WITH b AS (
    SELECT ict.stay_id, {dbin('t.charttime','ict.icu_intime')} AS bin,
        t.sodium, t.potassium, t.bicarbonate, t.bun, t.creatinine,
        t.glucose, t.albumin, t.aniongap, t.calcium
    FROM chemistry t
    INNER JOIN icustay_detail ict ON t.hadm_id = ict.hadm_id
    WHERE ict.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND t.charttime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, t.charttime) BETWEEN 0 AND 71)
SELECT stay_id, bin,
    ROUND(AVG(sodium),2)      AS sodium_mean,
    ROUND(AVG(potassium),2)   AS k_mean,
    ROUND(AVG(bicarbonate),2) AS bicarb_mean, ROUND(MIN(bicarbonate),2) AS bicarb_min,
    ROUND(AVG(bun),2)         AS bun_mean,    ROUND(MAX(bun),2)         AS bun_max,
    ROUND(AVG(creatinine),2)  AS cr_mean,     ROUND(MAX(creatinine),2)  AS cr_max,
    ROUND(AVG(glucose),2)     AS glucose_mean,
    ROUND(AVG(albumin),2)     AS albumin_mean,
    ROUND(AVG(aniongap),2)    AS ag_mean,     ROUND(MAX(aniongap),2)    AS ag_max,
    ROUND(AVG(calcium),2)     AS ca_mean
FROM b GROUP BY stay_id, bin
""").fetchdf()
    print(f"  chemistry: {len(temporal_frames['chemistry']):,} rows")
except Exception as e:
    temporal_frames['chemistry'] = pd.DataFrame(); print(f"  chemistry FAILED: {e}")

# ── CBC + DIFFERENTIAL (hadm_id) — fixed: neutrophils/lymphocytes (not _pct) ──
print("CBC + differential...")
try:
    temporal_frames['cbc'] = duck.execute(f"""
WITH cbc AS (
    SELECT ict.stay_id, {dbin('t.charttime','ict.icu_intime')} AS bin,
        t.hematocrit, t.hemoglobin, t.platelet, t.wbc
    FROM complete_blood_count t
    INNER JOIN icustay_detail ict ON t.hadm_id = ict.hadm_id
    WHERE ict.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND t.charttime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, t.charttime) BETWEEN 0 AND 71),
dif AS (
    SELECT ict.stay_id, {dbin('t.charttime','ict.icu_intime')} AS bin,
        t.neutrophils AS neut_pct, t.lymphocytes AS lymph_pct
    FROM blood_differential t
    INNER JOIN icustay_detail ict ON t.hadm_id = ict.hadm_id
    WHERE ict.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND t.charttime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, t.charttime) BETWEEN 0 AND 71),
ca AS (
    SELECT stay_id, bin,
        ROUND(AVG(hematocrit),2) AS hct_mean, ROUND(MIN(hematocrit),2) AS hct_min,
        ROUND(AVG(hemoglobin),2) AS hgb_mean, ROUND(MIN(hemoglobin),2) AS hgb_min,
        ROUND(AVG(platelet),2)   AS plt_mean, ROUND(MIN(platelet),2)   AS plt_min,
        ROUND(AVG(wbc),2)        AS wbc_mean, ROUND(MAX(wbc),2)        AS wbc_max
    FROM cbc GROUP BY stay_id, bin),
da AS (
    SELECT stay_id, bin,
        ROUND(AVG(neut_pct),2)  AS neut_pct_mean,
        ROUND(AVG(lymph_pct),2) AS lymph_pct_mean,
        ROUND(AVG(CASE WHEN lymph_pct > 0 THEN neut_pct / lymph_pct END),2) AS nlr_mean
    FROM dif GROUP BY stay_id, bin)
SELECT c.stay_id, c.bin,
    c.hct_mean, c.hct_min, c.hgb_mean, c.hgb_min,
    c.plt_mean, c.plt_min, c.wbc_mean, c.wbc_max,
    d.neut_pct_mean, d.lymph_pct_mean, d.nlr_mean
FROM ca c LEFT JOIN da d USING (stay_id, bin)
""").fetchdf()
    print(f"  cbc+diff: {len(temporal_frames['cbc']):,} rows")
except Exception as e:
    temporal_frames['cbc'] = pd.DataFrame(); print(f"  cbc+diff FAILED: {e}")

# ── COAGULATION (hadm_id) ─────────────────────────────────────────────────────
print("Coagulation...")
try:
    temporal_frames['coag'] = duck.execute(f"""
WITH b AS (
    SELECT ict.stay_id, {dbin('t.charttime','ict.icu_intime')} AS bin,
        t.inr, t.pt, t.ptt, t.fibrinogen
    FROM coagulation t
    INNER JOIN icustay_detail ict ON t.hadm_id = ict.hadm_id
    WHERE ict.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND t.charttime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, t.charttime) BETWEEN 0 AND 71)
SELECT stay_id, bin,
    ROUND(AVG(inr),2)        AS inr_mean,       ROUND(MAX(inr),2)        AS inr_max,
    ROUND(AVG(pt),2)         AS pt_mean,
    ROUND(AVG(ptt),2)        AS ptt_mean,       ROUND(MAX(ptt),2)        AS ptt_max,
    ROUND(AVG(fibrinogen),2) AS fibrinogen_mean
FROM b GROUP BY stay_id, bin
""").fetchdf()
    print(f"  coagulation: {len(temporal_frames['coag']):,} rows")
except Exception as e:
    temporal_frames['coag'] = pd.DataFrame(); print(f"  coagulation FAILED: {e}")

# ── BLOOD GAS — arterial (hadm_id, uses pao2fio2ratio pre-computed) ───────────
print("Blood gas...")
try:
    temporal_frames['bg'] = duck.execute(f"""
WITH b AS (
    SELECT ict.stay_id, {dbin('t.charttime','ict.icu_intime')} AS bin,
        t.ph, t.pco2, t.po2, t.lactate, t.baseexcess, t.pao2fio2ratio
    FROM bg t
    INNER JOIN icustay_detail ict ON t.hadm_id = ict.hadm_id
    WHERE ict.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND UPPER(COALESCE(t.specimen,'')) LIKE '%ART%'
      AND t.charttime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, t.charttime) BETWEEN 0 AND 71)
SELECT stay_id, bin,
    ROUND(AVG(ph),3)            AS ph_mean,       ROUND(MIN(ph),3)            AS ph_min,
    ROUND(AVG(pco2),2)          AS pco2_mean,
    ROUND(AVG(po2),2)           AS po2_mean,
    ROUND(AVG(lactate),2)       AS lactate_mean,  ROUND(MAX(lactate),2)       AS lactate_max,
    ROUND(AVG(baseexcess),2)    AS be_mean,
    ROUND(AVG(pao2fio2ratio),2) AS pf_ratio_mean, ROUND(MIN(pao2fio2ratio),2) AS pf_ratio_min
FROM b GROUP BY stay_id, bin
""").fetchdf()
    print(f"  blood gas: {len(temporal_frames['bg']):,} rows")
except Exception as e:
    temporal_frames['bg'] = pd.DataFrame(); print(f"  blood gas FAILED: {e}")

# ── CARDIAC MARKERS (hadm_id) — troponin_i removed (not in DuckDB) ────────────
print("Cardiac markers...")
try:
    temporal_frames['cardiac'] = duck.execute(f"""
WITH b AS (
    SELECT ict.stay_id, {dbin('t.charttime','ict.icu_intime')} AS bin,
        t.troponin_t, t.ck_mb, t.ntprobnp
    FROM cardiac_marker t
    INNER JOIN icustay_detail ict ON t.hadm_id = ict.hadm_id
    WHERE ict.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND t.charttime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, t.charttime) BETWEEN 0 AND 71)
SELECT stay_id, bin,
    ROUND(MAX(troponin_t),4) AS trop_t_max,
    ROUND(MAX(ck_mb),2)      AS ckmb_max,
    ROUND(MAX(ntprobnp),2)   AS ntprobnp_max
FROM b GROUP BY stay_id, bin
""").fetchdf()
    print(f"  cardiac markers: {len(temporal_frames['cardiac']):,} rows")
except Exception as e:
    temporal_frames['cardiac'] = pd.DataFrame(); print(f"  cardiac markers FAILED: {e}")

# ── LIVER ENZYMES (hadm_id) ───────────────────────────────────────────────────
print("Liver enzymes...")
try:
    temporal_frames['enzymes'] = duck.execute(f"""
WITH b AS (
    SELECT ict.stay_id, {dbin('t.charttime','ict.icu_intime')} AS bin,
        t.alt, t.ast, t.alp, t.bilirubin_total
    FROM enzyme t
    INNER JOIN icustay_detail ict ON t.hadm_id = ict.hadm_id
    WHERE ict.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND t.charttime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, t.charttime) BETWEEN 0 AND 71)
SELECT stay_id, bin,
    ROUND(AVG(alt),2)            AS alt_mean,  ROUND(MAX(alt),2)            AS alt_max,
    ROUND(AVG(ast),2)            AS ast_mean,  ROUND(MAX(ast),2)            AS ast_max,
    ROUND(AVG(alp),2)            AS alp_mean,
    ROUND(AVG(bilirubin_total),2) AS bili_mean, ROUND(MAX(bilirubin_total),2) AS bili_max
FROM b GROUP BY stay_id, bin
""").fetchdf()
    print(f"  enzymes: {len(temporal_frames['enzymes']):,} rows")
except Exception as e:
    temporal_frames['enzymes'] = pd.DataFrame(); print(f"  enzymes FAILED: {e}")

# ── SOFA per bin (stay_id) ────────────────────────────────────────────────────
print("SOFA per bin...")
try:
    temporal_frames['sofa'] = duck.execute(f"""
WITH b AS (
    SELECT s.stay_id,
        CASE WHEN datediff('hour', ict.icu_intime, s.endtime) < 24 THEN 0
             WHEN datediff('hour', ict.icu_intime, s.endtime) < 48 THEN 1
             ELSE 2 END AS bin,
        s.sofa_24hours AS sofa_score
    FROM sofa s
    INNER JOIN icustay_detail ict ON s.stay_id = ict.stay_id
    WHERE s.stay_id IN (SELECT stay_id FROM _cohort_stays)
      AND s.endtime >= ict.icu_intime
      AND datediff('hour', ict.icu_intime, s.endtime) BETWEEN 0 AND 71)
SELECT stay_id, bin,
    ROUND(AVG(sofa_score),2) AS sofa_mean, ROUND(MAX(sofa_score),2) AS sofa_max
FROM b GROUP BY stay_id, bin
""").fetchdf()
    print(f"  SOFA: {len(temporal_frames['sofa']):,} rows")
except Exception as e:
    temporal_frames['sofa'] = pd.DataFrame(); print(f"  SOFA FAILED: {e}")

# ── VENTILATION per bin (stay_id, interval overlap) ──────────────────────────
print("Ventilation...")
try:
    temporal_frames['vent'] = duck.execute("""
WITH bins(bin, h0, h1) AS (VALUES (0, 0, 24), (1, 24, 48), (2, 48, 72))
SELECT ict.stay_id, b.bin,
    MAX(CASE WHEN v.ventilation_status = 'InvasiveVent'
        AND v.starttime < ict.icu_intime + INTERVAL (b.h1) HOUR
        AND COALESCE(v.endtime, ict.icu_outtime) > ict.icu_intime + INTERVAL (b.h0) HOUR
        THEN 1 ELSE 0 END) AS vent_invasive,
    MAX(CASE WHEN v.ventilation_status IN ('NonInvasiveVent','HighFlow')
        AND v.starttime < ict.icu_intime + INTERVAL (b.h1) HOUR
        AND COALESCE(v.endtime, ict.icu_outtime) > ict.icu_intime + INTERVAL (b.h0) HOUR
        THEN 1 ELSE 0 END) AS vent_noninvasive
FROM icustay_detail ict
CROSS JOIN bins b
LEFT JOIN ventilation v ON ict.stay_id = v.stay_id
WHERE ict.stay_id IN (SELECT stay_id FROM _cohort_stays)
GROUP BY ict.stay_id, b.bin
""").fetchdf()
    print(f"  ventilation: {len(temporal_frames['vent']):,} rows")
except Exception as e:
    temporal_frames['vent'] = pd.DataFrame(); print(f"  ventilation FAILED: {e}")

# ── RRT per bin (stay_id) ─────────────────────────────────────────────────────
print("RRT...")
try:
    temporal_frames['rrt'] = duck.execute("""
WITH bins(bin, h0, h1) AS (VALUES (0, 0, 24), (1, 24, 48), (2, 48, 72))
SELECT ict.stay_id, b.bin,
    MAX(COALESCE(r.dialysis_active, 0)) AS rrt_flag
FROM icustay_detail ict
CROSS JOIN bins b
LEFT JOIN rrt r
    ON ict.stay_id = r.stay_id
    AND r.charttime >= ict.icu_intime + INTERVAL (b.h0) HOUR
    AND r.charttime <  ict.icu_intime + INTERVAL (b.h1) HOUR
WHERE ict.stay_id IN (SELECT stay_id FROM _cohort_stays)
GROUP BY ict.stay_id, b.bin
""").fetchdf()
    print(f"  rrt: {len(temporal_frames['rrt']):,} rows")
except Exception as e:
    temporal_frames['rrt'] = pd.DataFrame(); print(f"  rrt FAILED: {e}")

print("\n✓ Temporal extraction complete")
print(f"  Successful: {[k for k,v in temporal_frames.items() if not v.empty]}")
print(f"  Failed:     {[k for k,v in temporal_frames.items() if v.empty]}")


# STEP 1.4: FEATURE MERGE
# Merge all static derived features onto cohort_df.
# Also recompute readmissions / fragmentation and keep procedures_df for EDA.

In [ ]:
# STEP 1.4: FEATURE MERGE
# Merge all static derived features onto cohort_df.
# Also recompute readmissions / fragmentation and keep procedures_df for EDA.

# Static merges
for df, key, label in [
    (charlson_df, 'hadm_id', 'charlson'),
    (anthro_df,   'stay_id', 'anthropometrics'),
    (severity_df, 'stay_id', 'severity_scores'),
]:
    if not df.empty:
        cohort_df = cohort_df.merge(df, on=key, how='left')
        print(f"  merged {label}")

for df, key, label in [
    (services_df, 'hadm_id', 'services'),
    (sepsis_df,   'stay_id', 'sepsis3'),
    (aki_df,      'stay_id', 'aki'),
]:
    if not df.empty:
        cohort_df = cohort_df.merge(df, on=key, how='left')
        print(f"  merged {label}")

# Readmissions / fragmentation
readm = (cohort_df.groupby('hadm_id')
         .agg(num_icu_stays=('stay_id','count')).reset_index())
readm['has_fragmentation'] = (readm['num_icu_stays'] > 1).astype(int)
cohort_df = cohort_df.merge(readm, on='hadm_id', how='left')

# Procedures (ICD codes — for EDA only, NOT model features)
try:
    hadm_ids_proc = cohort_df[['hadm_id','stay_id']].drop_duplicates()
    proc_raw = run_chunked(lambda ids: f"""
        SELECT pr.hadm_id, di.long_title AS procedure_name, COUNT(*) AS n
        FROM `physionet-data.mimiciv_3_1_hosp.procedures_icd` pr
        LEFT JOIN `physionet-data.mimiciv_3_1_hosp.d_icd_procedures` di
            ON pr.icd_code = di.icd_code AND pr.icd_version = di.icd_version
        WHERE pr.hadm_id IN ({ids})
        GROUP BY pr.hadm_id, di.long_title
    """, cohort_df['hadm_id'].drop_duplicates().tolist())
    procedures_df = proc_raw.merge(hadm_ids_proc, on='hadm_id', how='left')
    print(f"  procedures (EDA only): {len(procedures_df):,} rows")
except Exception as e:
    procedures_df = pd.DataFrame()
    print(f"  procedures FAILED: {e}")

print(f"\ncohort_df final shape: {cohort_df.shape}")
print(f"  Columns: {list(cohort_df.columns)}")

# ── Export cohort_df snapshot ────────────────────────────────────────────────
CSV_PATH = '/content/drive/MyDrive/icu_cohort_snapshot.csv'
cohort_df.to_csv(CSV_PATH, index=False)
print(f"\n✓ cohort_df saved to {CSV_PATH}")
print(f"  Shape: {cohort_df.shape[0]:,} rows × {cohort_df.shape[1]} columns")
print(cohort_df.dtypes.to_string())

# STEP 2.1: CREATE STRATIFIED COHORTS

In [ ]:

print("\\nCreating stratified analysis cohorts...\\n")

cohort_df['cohort_group'] = pd.cut(
    cohort_df['los_days'],
    bins=[0, 21, 1000],
    labels=['<21 days', '21+ days'],
    right=False
)

cohort_df['outcome_label'] = cohort_df['hospital_expire_flag'].map({0: 'Survived', 1: 'Died'})
cohort_df['analysis_group'] = (
    cohort_df['cohort_group'].astype(str) + ' - ' + cohort_df['outcome_label'].astype(str)
)

print(f"Analysis Group Distribution:")
group_counts = cohort_df['analysis_group'].value_counts().sort_index()
for group, count in group_counts.items():
    pct = 100 * count / len(cohort_df)
    print(f"  {group:30s}: {count:5d} ({pct:5.1f}%)")

# STEP 2.2: INSURANCE TYPE & MORTALITY ANALYSIS — 4-Group Comparison

In [ ]:

print("\n=== INSURANCE TYPE & MORTALITY ANALYSIS (4-Group) ===")

GROUP_ORDER  = ['21+ days - Died', '21+ days - Survived', '<21 days - Died', '<21 days - Survived']
GROUP_COLORS = {
    '21+ days - Died':      '#c0392b',
    '21+ days - Survived':  '#e67e22',
    '<21 days - Died':      '#2980b9',
    '<21 days - Survived':  '#27ae60',
}

# ── Tabular summary ──────────────────────────────────────────────────────────
ins_grp = (cohort_df.groupby(['insurance', 'analysis_group'])
                    .agg(n=('stay_id', 'count'),
                         deaths=('hospital_expire_flag', 'sum'),
                         mean_los=('los_days', 'mean'))
                    .reset_index())
ins_grp['mortality_pct'] = (ins_grp['deaths'] / ins_grp['n'] * 100).round(1)
ins_grp['mean_los']      = ins_grp['mean_los'].round(1)
print(ins_grp.to_string(index=False))

# ── Pivot tables ─────────────────────────────────────────────────────────────
def _pivot(df, val):
    p = df.pivot_table(index='insurance', columns='analysis_group', values=val, fill_value=0)
    return p.reindex(columns=[g for g in GROUP_ORDER if g in p.columns])

pivot_mort = _pivot(ins_grp, 'mortality_pct')
pivot_n    = _pivot(ins_grp, 'n')

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Insurance Type Analysis — 4-Group Stratification',
             fontsize=14, fontweight='bold')

def _grouped_barh(ax, pivot, xlabel, title):
    """Horizontal grouped bar chart from a pivot table."""
    n_groups = len(pivot.columns)
    bar_w    = 0.8 / n_groups
    y_pos    = np.arange(len(pivot.index))
    for j, grp in enumerate(pivot.columns):
        offset = (j - n_groups / 2 + 0.5) * bar_w
        ax.bar(y_pos + offset, pivot[grp], width=bar_w,
               label=grp, color=GROUP_COLORS.get(grp, '#888'),
               alpha=0.85, edgecolor='white')
    ax.set_xticks(y_pos)
    ax.set_xticklabels(pivot.index, rotation=30, ha='right')
    ax.set_ylabel(xlabel, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=7, title='Group')
    ax.grid(axis='y', alpha=0.3)

_grouped_barh(axes[0], pivot_mort, 'Mortality Rate (%)',   'Mortality Rate by Insurance & Group')
_grouped_barh(axes[1], pivot_n,    'Number of ICU Stays',  'Volume by Insurance & Group')

# Chart 3: Box plot LOS × insurance × group ──────────────────────────────────
from matplotlib.patches import Patch
los_clip = cohort_df[cohort_df['los_days'] <= cohort_df['los_days'].quantile(0.99)].copy()
ins_types = sorted(los_clip['insurance'].dropna().unique())
gap = len(GROUP_ORDER) + 1
ins_centers = []
for i_ins, ins in enumerate(ins_types):
    base = i_ins * gap
    ins_centers.append(base + (len(GROUP_ORDER) - 1) / 2)
    for j_grp, grp in enumerate(GROUP_ORDER):
        vals = los_clip[(los_clip['insurance'] == ins) &
                        (los_clip['analysis_group'] == grp)]['los_days'].dropna()
        if len(vals) < 10:
            continue
        bp = axes[2].boxplot(vals, positions=[base + j_grp], widths=0.65,
                             patch_artist=True,
                             medianprops=dict(color='black', linewidth=1.5),
                             whiskerprops=dict(linewidth=0.7),
                             capprops=dict(linewidth=0.7),
                             flierprops=dict(marker='.', markersize=2, alpha=0.3))
        for patch in bp['boxes']:
            patch.set_facecolor(GROUP_COLORS.get(grp, '#888'))
            patch.set_alpha(0.75)
axes[2].set_xticks(ins_centers)
axes[2].set_xticklabels(ins_types, rotation=30, ha='right')
axes[2].set_ylabel('LOS (days)', fontweight='bold')
axes[2].set_title('LOS Distribution by Insurance & Group', fontsize=11, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)
present_groups = [g for g in GROUP_ORDER if g in cohort_df['analysis_group'].unique()]
axes[2].legend(
    handles=[Patch(facecolor=GROUP_COLORS[g], label=g) for g in present_groups],
    fontsize=7, title='Group', loc='upper right'
)

plt.tight_layout()
plt.savefig('01_insurance_mortality_4group.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✓ Saved: 01_insurance_mortality_4group.png")


# STEP 2.3: DIAGNOSIS & LOS ANALYSIS — 4-Group Comparison

In [ ]:
# STEP 2.3: DIAGNOSIS & LOS ANALYSIS — 4-Group Comparison
print("\nAnalyzing diagnoses & LOS (4-group)...\n")

GROUP_ORDER  = ['21+ days - Died', '21+ days - Survived', '<21 days - Died', '<21 days - Survived']
GROUP_COLORS = {
    '21+ days - Died':      '#c0392b',
    '21+ days - Survived':  '#e67e22',
    '<21 days - Died':      '#2980b9',
    '<21 days - Survived':  '#27ae60',
}

hadm_ids_list = cohort_df['hadm_id'].drop_duplicates().tolist()

diagnosis_query = f"""
SELECT ie.stay_id, dx.icd_code, dx.icd_version, d.long_title AS diagnosis_name
FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` dx
LEFT JOIN `physionet-data.mimiciv_3_1_hosp.d_icd_diagnoses` d
    ON dx.icd_code = d.icd_code AND dx.icd_version = d.icd_version
INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
    ON dx.hadm_id = ie.hadm_id
WHERE dx.hadm_id IN ({','.join(map(str, hadm_ids_list[:5000]))})
"""

try:
    diagnoses_df = client.query(diagnosis_query).to_dataframe()

    dx_los = diagnoses_df.merge(
        cohort_df[['stay_id', 'los_days', 'analysis_group']], on='stay_id'
    )

    # Top 15 diagnoses by total stay count
    top15_dx = (dx_los.groupby('diagnosis_name')['stay_id']
                      .count()
                      .sort_values(ascending=False)
                      .head(15).index.tolist())

    dx_top = dx_los[dx_los['diagnosis_name'].isin(top15_dx)].copy()
    dx_top['diagnosis_name'] = dx_top['diagnosis_name'].str[:52]

    # ── Pivot 1: mean LOS per diagnosis × group ───────────────────────────────
    dx_mean_los = (dx_top.groupby(['diagnosis_name', 'analysis_group'])['los_days']
                         .mean().unstack(fill_value=np.nan))
    dx_mean_los = dx_mean_los.reindex(
        columns=[g for g in GROUP_ORDER if g in dx_mean_los.columns]
    )
    sort_idx = dx_top.groupby('diagnosis_name')['los_days'].mean().sort_values(ascending=True).index
    dx_mean_los = dx_mean_los.reindex(sort_idx)

    # ── Pivot 2: % of admissions per group per diagnosis ─────────────────────
    dx_cnt = (dx_top.groupby(['diagnosis_name', 'analysis_group'])['stay_id']
                    .count().unstack(fill_value=0))
    dx_cnt = dx_cnt.reindex(columns=[g for g in GROUP_ORDER if g in dx_cnt.columns], fill_value=0)
    dx_pct = dx_cnt.div(dx_cnt.sum(axis=1), axis=0) * 100
    dx_pct = dx_pct.reindex(dx_mean_los.index)

    fig, axes = plt.subplots(1, 2, figsize=(22, 10))
    fig.suptitle('Diagnosis Analysis — 4-Group Stratification (Top 15 by Volume)',
                 fontsize=14, fontweight='bold')

    # Chart 1: grouped horizontal bar — mean LOS
    y_pos = np.arange(len(dx_mean_los.index))
    n_g   = len(dx_mean_los.columns)
    bar_h = 0.8 / n_g
    for j, grp in enumerate(dx_mean_los.columns):
        offset = (j - n_g / 2 + 0.5) * bar_h
        axes[0].barh(y_pos + offset, dx_mean_los[grp].fillna(0).values,
                     height=bar_h, label=grp,
                     color=GROUP_COLORS.get(grp, '#888'), alpha=0.85, edgecolor='white')
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(dx_mean_los.index, fontsize=8)
    axes[0].set_xlabel('Mean LOS (days)', fontweight='bold')
    axes[0].set_title('Mean LOS per Diagnosis by Group', fontsize=11, fontweight='bold')
    axes[0].legend(fontsize=8, title='Group')
    axes[0].grid(axis='x', alpha=0.3)

    # Chart 2: stacked % bar — admission mix
    colors_list = [GROUP_COLORS.get(g, '#888') for g in dx_pct.columns]
    dx_pct.plot(kind='barh', stacked=True, ax=axes[1],
                color=colors_list, alpha=0.85, edgecolor='white', legend=False)
    axes[1].set_xlabel('% of Admissions in Diagnosis', fontweight='bold')
    axes[1].set_ylabel('')
    axes[1].set_yticklabels(dx_pct.index, fontsize=8)
    axes[1].set_title('Admission Mix per Diagnosis (% by Group)', fontsize=11, fontweight='bold')
    axes[1].axvline(x=100, color='black', linewidth=0.5, linestyle='--')
    axes[1].grid(axis='x', alpha=0.3)
    from matplotlib.patches import Patch
    axes[1].legend(
        handles=[Patch(facecolor=GROUP_COLORS[g], label=g) for g in dx_pct.columns],
        fontsize=8, title='Group', loc='lower right'
    )

    plt.tight_layout()
    plt.savefig('02_diagnosis_4group.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("\n✓ Saved: 02_diagnosis_4group.png")

except Exception as e:
    print(f"⚠ Warning: {e}")


# STEP 2.4: PROCEDURE BALANCE COMPARISON — 4-Group

In [ ]:

print("\nAnalyzing procedure balance (4-group)...\n")

GROUP_ORDER  = ['21+ days - Died', '21+ days - Survived', '<21 days - Died', '<21 days - Survived']
GROUP_COLORS = {
    '21+ days - Died':      '#c0392b',
    '21+ days - Survived':  '#e67e22',
    '<21 days - Died':      '#2980b9',
    '<21 days - Survived':  '#27ae60',
}

if len(procedures_df) > 0:
    proc_analysis = procedures_df.merge(
        cohort_df[['stay_id', 'los_days', 'icu_stay_21plus',
                   'cohort_group', 'analysis_group']],
        on='stay_id'
    )

    # Group sizes as denominators
    grp_sizes = cohort_df['analysis_group'].value_counts().to_dict()

    # Unique stays per procedure × group
    proc_grp = (proc_analysis.groupby(['procedure_name', 'analysis_group'])['stay_id']
                              .nunique().unstack(fill_value=0))
    proc_grp = proc_grp.reindex(
        columns=[g for g in GROUP_ORDER if g in proc_grp.columns], fill_value=0
    ).astype(float)

    # Utilisation % within each group
    proc_pct = proc_grp.copy()
    for grp in proc_pct.columns:
        proc_pct[grp] = (proc_pct[grp] / grp_sizes.get(grp, 1) * 100).round(2)

    # Top 15 by mean utilisation
    top15_proc = proc_pct.mean(axis=1).sort_values(ascending=False).head(15).index
    proc_top   = proc_pct.loc[top15_proc].sort_values(
        proc_pct.columns[0] if len(proc_pct.columns) else proc_pct.columns[0],
        ascending=True
    )

    # Scatter data — split by outcome within each LOS group
    n = {g: grp_sizes.get(g, 1) for g in GROUP_ORDER}
    scatter_df = proc_grp.reindex(top15_proc).copy()
    for col in GROUP_ORDER:
        if col not in scatter_df.columns:
            scatter_df[col] = 0
    scatter_df['pct_21plus_died'] = (scatter_df['21+ days - Died']    / n['21+ days - Died']    * 100).round(2)
    scatter_df['pct_21plus_surv'] = (scatter_df['21+ days - Survived']/ n['21+ days - Survived']* 100).round(2)
    scatter_df['pct_lt_died']     = (scatter_df['<21 days - Died']     / n['<21 days - Died']    * 100).round(2)
    scatter_df['pct_lt_surv']     = (scatter_df['<21 days - Survived'] / n['<21 days - Survived']* 100).round(2)

    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.suptitle('Procedure Utilisation — 4-Group Stratification',
                 fontsize=14, fontweight='bold')

    # Chart 1: grouped horizontal bar
    y_pos = np.arange(len(proc_top.index))
    n_g   = len(proc_top.columns)
    bar_h = 0.8 / n_g
    for j, grp in enumerate(proc_top.columns):
        offset = (j - n_g / 2 + 0.5) * bar_h
        axes[0].barh(y_pos + offset, proc_top[grp].values,
                     height=bar_h, label=grp,
                     color=GROUP_COLORS.get(grp, '#888'), alpha=0.85, edgecolor='white')
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels([p[:45] for p in proc_top.index], fontsize=8)
    axes[0].set_xlabel('Utilisation (% of group)', fontweight='bold')
    axes[0].set_title('Top 15 Procedures by Group', fontsize=11, fontweight='bold')
    axes[0].legend(fontsize=7, title='Group')
    axes[0].grid(axis='x', alpha=0.3)

    # Chart 2: 21+ vs <21 scatter, coloured by outcome
    axes[1].scatter(scatter_df['pct_lt_surv'],  scatter_df['pct_21plus_surv'],
                    color=GROUP_COLORS['21+ days - Survived'], s=90, alpha=0.8,
                    label='Survived', zorder=3)
    axes[1].scatter(scatter_df['pct_lt_died'],  scatter_df['pct_21plus_died'],
                    color=GROUP_COLORS['21+ days - Died'],    s=90, alpha=0.8,
                    marker='^', label='Died', zorder=3)
    all_vals = scatter_df[['pct_lt_surv','pct_lt_died','pct_21plus_surv','pct_21plus_died']].values.flatten()
    max_val  = max(all_vals.max() if len(all_vals) else 1, 1)
    axes[1].plot([0, max_val], [0, max_val], 'k--', alpha=0.3, linewidth=1.2, label='Equal use')
    axes[1].set_xlabel('Utilisation in <21 Day Group (%)', fontweight='bold')
    axes[1].set_ylabel('Utilisation in 21+ Day Group (%)', fontweight='bold')
    axes[1].set_title('Procedure Utilisation: 21+ vs <21\n(△ = Died  ● = Survived)',
                      fontsize=11, fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('03_procedure_4group.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("\n✓ Saved: 03_procedure_4group.png")
else:
    print("⚠ No procedure data available")


# STEP 2.5: READMISSION & FRAGMENTATION — 4-Group

In [ ]:

print("\nAnalyzing readmission patterns (4-group)...\n")

GROUP_ORDER  = ['21+ days - Died', '21+ days - Survived', '<21 days - Died', '<21 days - Survived']
GROUP_COLORS = {
    '21+ days - Died':      '#c0392b',
    '21+ days - Survived':  '#e67e22',
    '<21 days - Died':      '#2980b9',
    '<21 days - Survived':  '#27ae60',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('ICU Readmission & Fragmentation — 4-Group Analysis',
             fontsize=14, fontweight='bold')

# [0,0]: Fragmentation rate by all 4 analysis_group values ───────────────────
present = [g for g in GROUP_ORDER if g in cohort_df['analysis_group'].unique()]
frag_4g = (cohort_df.groupby('analysis_group')['has_fragmentation']
                     .mean() * 100).reindex(present)
bar_colors = [GROUP_COLORS.get(g, '#888') for g in frag_4g.index]
axes[0, 0].bar(range(len(frag_4g)), frag_4g.values,
               color=bar_colors, alpha=0.85, edgecolor='white')
for i, v in enumerate(frag_4g.values):
    axes[0, 0].text(i, v + 0.4, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
axes[0, 0].set_xticks(range(len(frag_4g)))
axes[0, 0].set_xticklabels(frag_4g.index, rotation=18, ha='right', fontsize=8)
axes[0, 0].set_ylabel('Fragmentation Rate (%)', fontweight='bold')
axes[0, 0].set_title('Fragmentation Rate by 4-Group', fontsize=11, fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)

# [0,1]: Mortality by fragmentation × cohort_group (unchanged — informative) ─
mort_frag = (cohort_df.groupby(['cohort_group', 'has_fragmentation'])
                       ['hospital_expire_flag'].mean() * 100)
mort_frag.unstack().plot(kind='bar', ax=axes[0, 1],
                          color=['#3498db', '#e67e22'], width=0.7, edgecolor='white')
axes[0, 1].set_ylabel('Mortality Rate (%)', fontweight='bold')
axes[0, 1].set_title('Mortality by LOS Group & Fragmentation',
                     fontsize=11, fontweight='bold')
axes[0, 1].legend(['Single Stay', 'Fragmented'], title='Stay Type')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)
axes[0, 1].grid(axis='y', alpha=0.3)

# [1,0]: Total LOS vs number of ICU stays (unchanged) ───────────────────────
los_by_frags = cohort_df.groupby('num_icu_stays')['los_days'].agg(['mean', 'median', 'count'])
los_by_frags = los_by_frags[los_by_frags['count'] >= 5]
axes[1, 0].bar(range(len(los_by_frags)), los_by_frags['mean'],
               color='steelblue', alpha=0.7, label='Mean')
axes[1, 0].scatter(range(len(los_by_frags)), los_by_frags['median'],
                   color='darkred', s=150, zorder=5, label='Median')
axes[1, 0].set_xticks(range(len(los_by_frags)))
axes[1, 0].set_xticklabels(los_by_frags.index)
axes[1, 0].set_xlabel('Number of ICU Stays', fontweight='bold')
axes[1, 0].set_ylabel('Length of Stay (days)', fontweight='bold')
axes[1, 0].set_title('Total LOS by Number of Fragments', fontsize=11, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# [1,1]: Updated summary — all 4 groups ─────────────────────────────────────
axes[1, 1].axis('off')
lines = ["4-Group Summary Statistics\n"]
for grp in GROUP_ORDER:
    sub = cohort_df[cohort_df['analysis_group'] == grp]
    if len(sub) == 0:
        continue
    lines += [
        f"{grp}",
        f"  N:           {len(sub):,}",
        f"  Mortality:   {sub['hospital_expire_flag'].mean():.1%}",
        f"  Fragmented:  {sub['has_fragmentation'].mean():.1%}",
        f"  Median LOS:  {sub['los_days'].median():.1f} d",
        "",
    ]
axes[1, 1].text(
    0.05, 0.97, "\n".join(lines),
    transform=axes[1, 1].transAxes, fontsize=10,
    verticalalignment='top', fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
)

plt.tight_layout()
plt.savefig('04_fragmentation_4group.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 04_fragmentation_4group.png")


# STEP 2.6: EXTENDED 4-GROUP EDA — Demographics, Severity, Trajectory, Outcomes

In [ ]:

print("\nExtended 4-group EDA...\n")

GROUP_ORDER  = ['21+ days - Died', '21+ days - Survived', '<21 days - Died', '<21 days - Survived']
GROUP_COLORS = {
    '21+ days - Died':      '#c0392b',
    '21+ days - Survived':  '#e67e22',
    '<21 days - Died':      '#2980b9',
    '<21 days - Survived':  '#27ae60',
}
palette = [GROUP_COLORS[g] for g in GROUP_ORDER]

df = cohort_df.copy()
df['analysis_group'] = pd.Categorical(df['analysis_group'], categories=GROUP_ORDER, ordered=True)
grp_sizes = df['analysis_group'].value_counts().reindex(GROUP_ORDER)

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1: DEMOGRAPHICS
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle('Demographics by 4-Group', fontsize=15, fontweight='bold')

# 1a. Age violin
grp_data = [df[df['analysis_group']==g]['age_at_icu'].dropna() for g in GROUP_ORDER]
vp = axes[0,0].violinplot(grp_data, positions=range(4), showmedians=True, showextrema=True)
for i, (pc, color) in enumerate(zip(vp['bodies'], palette)):
    pc.set_facecolor(color); pc.set_alpha(0.7)
axes[0,0].set_xticks(range(4)); axes[0,0].set_xticklabels([g.replace(' - ','\n') for g in GROUP_ORDER], fontsize=9)
axes[0,0].set_ylabel('Age (years)'); axes[0,0].set_title('Age Distribution', fontweight='bold')
axes[0,0].grid(axis='y', alpha=0.3)
for i, d in enumerate(grp_data):
    axes[0,0].text(i, d.median(), f' {d.median():.0f}', va='center', fontsize=8, fontweight='bold')

# 1b. Sex breakdown
sex_pct = df.groupby('analysis_group')['gender'].value_counts(normalize=True).unstack(fill_value=0) * 100
sex_pct = sex_pct.reindex(GROUP_ORDER)
x = np.arange(len(GROUP_ORDER)); w = 0.35
if 'M' in sex_pct.columns:
    axes[0,1].bar(x - w/2, sex_pct['M'], w, label='Male',   color='steelblue', alpha=0.8)
if 'F' in sex_pct.columns:
    axes[0,1].bar(x + w/2, sex_pct['F'], w, label='Female', color='salmon',    alpha=0.8)
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels([g.replace(' - ','\n') for g in GROUP_ORDER], fontsize=9)
axes[0,1].set_ylabel('% of Group'); axes[0,1].set_title('Sex Breakdown', fontweight='bold')
axes[0,1].legend(); axes[0,1].grid(axis='y', alpha=0.3); axes[0,1].set_ylim(0, 100)

# 1c. Race breakdown
race_col = 'race' if 'race' in df.columns else None
if race_col:
    df['race_grp'] = df[race_col].apply(
        lambda x: x if str(x) in ['WHITE','BLACK/AFRICAN AMERICAN','HISPANIC/LATINO','ASIAN'] else 'OTHER')
    race_pct = df.groupby('analysis_group')['race_grp'].value_counts(normalize=True).unstack(fill_value=0)*100
    race_pct = race_pct.reindex(GROUP_ORDER).fillna(0)
    race_colors = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']
    bottom = np.zeros(len(GROUP_ORDER))
    for i, (col, color) in enumerate(zip(race_pct.columns, race_colors)):
        axes[0,2].bar(range(len(GROUP_ORDER)), race_pct[col], bottom=bottom, label=col, color=color, alpha=0.85)
        bottom += race_pct[col].values
    axes[0,2].set_xticks(range(4)); axes[0,2].set_xticklabels([g.replace(' - ','\n') for g in GROUP_ORDER], fontsize=9)
    axes[0,2].set_ylabel('% of Group'); axes[0,2].set_title('Race/Ethnicity Breakdown', fontweight='bold')
    axes[0,2].legend(fontsize=8, loc='lower right'); axes[0,2].set_ylim(0,100); axes[0,2].grid(axis='y', alpha=0.3)

# 1d. Insurance breakdown
ins_pct = df.groupby('analysis_group')['insurance'].value_counts(normalize=True).unstack(fill_value=0)*100
ins_pct = ins_pct.reindex(GROUP_ORDER).fillna(0)
ins_colors = plt.cm.Set2(np.linspace(0,1,len(ins_pct.columns)))
bottom = np.zeros(len(GROUP_ORDER))
for col, color in zip(ins_pct.columns, ins_colors):
    axes[1,0].bar(range(len(GROUP_ORDER)), ins_pct[col], bottom=bottom, label=col, color=color, alpha=0.85)
    bottom += ins_pct[col].values
axes[1,0].set_xticks(range(4)); axes[1,0].set_xticklabels([g.replace(' - ','\n') for g in GROUP_ORDER], fontsize=9)
axes[1,0].set_ylabel('% of Group'); axes[1,0].set_title('Insurance Type', fontweight='bold')
axes[1,0].legend(fontsize=8, loc='lower right'); axes[1,0].set_ylim(0,100); axes[1,0].grid(axis='y', alpha=0.3)

# 1e. Admission type
adm_pct = df.groupby('analysis_group')['admission_type'].value_counts(normalize=True).unstack(fill_value=0)*100
adm_pct = adm_pct.reindex(GROUP_ORDER).fillna(0)
adm_colors = plt.cm.Set1(np.linspace(0,1,len(adm_pct.columns)))
bottom = np.zeros(len(GROUP_ORDER))
for col, color in zip(adm_pct.columns, adm_colors):
    axes[1,1].bar(range(len(GROUP_ORDER)), adm_pct[col], bottom=bottom, label=col, color=color, alpha=0.85)
    bottom += adm_pct[col].values
axes[1,1].set_xticks(range(4)); axes[1,1].set_xticklabels([g.replace(' - ','\n') for g in GROUP_ORDER], fontsize=9)
axes[1,1].set_ylabel('% of Group'); axes[1,1].set_title('Admission Type', fontweight='bold')
axes[1,1].legend(fontsize=7, loc='lower right'); axes[1,1].set_ylim(0,100); axes[1,1].grid(axis='y', alpha=0.3)

# 1f. Care unit
cu_pct = df.groupby('analysis_group')['first_careunit'].value_counts(normalize=True).unstack(fill_value=0)*100
cu_pct = cu_pct.reindex(GROUP_ORDER).fillna(0)
cu_colors = plt.cm.tab10(np.linspace(0,1,len(cu_pct.columns)))
bottom = np.zeros(len(GROUP_ORDER))
for col, color in zip(cu_pct.columns, cu_colors):
    axes[1,2].bar(range(len(GROUP_ORDER)), cu_pct[col], bottom=bottom, label=col, color=color, alpha=0.85)
    bottom += cu_pct[col].values
axes[1,2].set_xticks(range(4)); axes[1,2].set_xticklabels([g.replace(' - ','\n') for g in GROUP_ORDER], fontsize=9)
axes[1,2].set_ylabel('% of Group'); axes[1,2].set_title('ICU Care Unit', fontweight='bold')
axes[1,2].legend(fontsize=7, loc='lower right'); axes[1,2].set_ylim(0,100); axes[1,2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_demographics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: eda_demographics.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2: CLINICAL SEVERITY AT ADMISSION
# ─────────────────────────────────────────────────────────────────────────────
severity_cols = {
    'sapsii_score':       'SAPSII',
    'apsiii_score':       'APSIII',
    'oasis_score':        'OASIS',
    'lods_score':         'LODS',
    'first_day_sofa':     'First-Day SOFA',
    'charlson_comorbidity_index': 'Charlson Index',
}
present_sev = {k:v for k,v in severity_cols.items() if k in df.columns}
n_sev = len(present_sev)
fig, axes = plt.subplots(2, max(n_sev//2, 1), figsize=(20, 10))
axes = axes.flatten()
fig.suptitle('Clinical Severity Scores at Admission — 4-Group', fontsize=15, fontweight='bold')

for ax, (col, label) in zip(axes, present_sev.items()):
    grp_data = [df[df['analysis_group']==g][col].dropna() for g in GROUP_ORDER]
    bp = ax.boxplot(grp_data, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], palette):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticklabels([g.replace(' - ','\n') for g in GROUP_ORDER], fontsize=8)
    ax.set_title(label, fontweight='bold', fontsize=11)
    ax.set_ylabel('Score'); ax.grid(axis='y', alpha=0.3)
    for i, d in enumerate(grp_data):
        if len(d):
            ax.text(i+1, d.median(), f' {d.median():.0f}', va='center', fontsize=8)

for ax in axes[len(present_sev):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig('eda_severity.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: eda_severity.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 3: KEY COMORBIDITIES (Charlson flags)
# ─────────────────────────────────────────────────────────────────────────────
comorbidity_cols = {
    'myocardial_infarct':          'Myocardial Infarct',
    'congestive_heart_failure':    'CHF',
    'chronic_pulmonary_disease':   'Chronic Pulmonary',
    'renal_disease':               'Renal Disease',
    'severe_liver_disease':        'Severe Liver',
    'metastatic_solid_tumor':      'Metastatic Tumor',
    'diabetes_with_cc':            'Diabetes w/ CC',
    'dementia':                    'Dementia',
    'cerebrovascular_disease':     'Cerebrovascular',
}
present_cm = {k:v for k,v in comorbidity_cols.items() if k in df.columns}
if present_cm:
    cm_pct = pd.DataFrame({
        label: df.groupby('analysis_group')[col].mean().reindex(GROUP_ORDER) * 100
        for col, label in present_cm.items()
    })
    fig, ax = plt.subplots(figsize=(16, 7))
    x = np.arange(len(cm_pct.columns)); w = 0.18
    for i, (grp, color) in enumerate(zip(GROUP_ORDER, palette)):
        offset = (i - 1.5) * w
        ax.bar(x + offset, cm_pct.loc[grp], w, label=grp, color=color, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(list(present_cm.values()), rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('% of Group with Comorbidity'); ax.set_title('Charlson Comorbidities by Group', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('eda_comorbidities.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Saved: eda_comorbidities.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 4: ICU INTERVENTIONS (sepsis, ventilation, RRT, AKI)
# ─────────────────────────────────────────────────────────────────────────────
intervention_cols = {
    'sepsis3_flag':         'Sepsis-3',
    'vent_invasive_b0':     'Invasive Vent (0–24h)',
    'vent_noninvasive_b0':  'Non-Inv Vent (0–24h)',
    'rrt_flag_b0':          'RRT (0–24h)',
    'aki_max_stage':        'AKI Stage ≥1',
}
present_iv = {k:v for k,v in intervention_cols.items() if k in df.columns}
if present_iv:
    iv_df = df.copy()
    if 'aki_max_stage' in iv_df.columns:
        iv_df['aki_max_stage'] = (iv_df['aki_max_stage'] >= 1).astype(float)
    iv_pct = pd.DataFrame({
        label: iv_df.groupby('analysis_group')[col].mean().reindex(GROUP_ORDER) * 100
        for col, label in present_iv.items()
    })
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(iv_pct.columns)); w = 0.18
    for i, (grp, color) in enumerate(zip(GROUP_ORDER, palette)):
        offset = (i - 1.5) * w
        ax.bar(x + offset, iv_pct.loc[grp], w, label=grp, color=color, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(list(present_iv.values()), fontsize=11)
    ax.set_ylabel('% of Group'); ax.set_title('ICU Interventions by Group', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('eda_interventions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Saved: eda_interventions.png")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 5: LAB TRAJECTORIES ACROSS 3 BINS (creatinine, lactate, P/F ratio)
# ─────────────────────────────────────────────────────────────────────────────
traj_cols = {
    'cr_mean':       'Creatinine (mean)',
    'lactate_mean':  'Lactate (mean)',
    'pf_ratio_mean': 'P/F Ratio (mean)',
}
present_traj = {k:v for k,v in traj_cols.items() if any(f'{k[:-4]}b{b}' in df.columns or k in df.columns for b in [0,1,2])}
# Build bin columns
traj_plot = {}
for base, label in traj_cols.items():
    stem = base[:-5] if base.endswith('_mean') else base
    cols_b = [f'{stem}_mean_b{b}' for b in [0,1,2]]
    cols_b2 = [f'{base}_b{b}' for b in [0,1,2]]
    if all(c in df.columns for c in cols_b):
        traj_plot[label] = cols_b
    elif all(c in df.columns for c in cols_b2):
        traj_plot[label] = cols_b2

if traj_plot:
    fig, axes = plt.subplots(1, len(traj_plot), figsize=(6*len(traj_plot), 5))
    if len(traj_plot) == 1: axes = [axes]
    fig.suptitle('Lab Trajectories Across 72h — 4-Group', fontsize=13, fontweight='bold')
    bin_labels = ['0–24h', '24–48h', '48–72h']
    for ax, (label, cols) in zip(axes, traj_plot.items()):
        for grp, color in zip(GROUP_ORDER, palette):
            means = [df[df['analysis_group']==grp][c].mean() for c in cols]
            sems  = [df[df['analysis_group']==grp][c].sem()  for c in cols]
            ax.plot(bin_labels, means, 'o-', color=color, lw=2, ms=7, label=grp)
            ax.fill_between(bin_labels,
                            [m-s for m,s in zip(means,sems)],
                            [m+s for m,s in zip(means,sems)],
                            color=color, alpha=0.15)
        ax.set_title(label, fontweight='bold'); ax.set_ylabel('Mean Value')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('eda_lab_trajectories.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Saved: eda_lab_trajectories.png")
else:
    print("  (Lab trajectory columns not found — skipping)")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 6: OUTCOMES — mortality, fragmentation, readmission
# ─────────────────────────────────────────────────────────────────────────────
outcome_cols = {
    'hospital_expire_flag': 'Hospital Mortality (%)',
    'has_fragmentation':    'Fragmented ICU Stay (%)',
    'num_icu_stays':        'Mean # ICU Stays',
}
present_out = {k:v for k,v in outcome_cols.items() if k in df.columns}
fig, axes = plt.subplots(1, len(present_out), figsize=(5*len(present_out), 5))
if len(present_out) == 1: axes = [axes]
fig.suptitle('Outcomes by 4-Group', fontsize=13, fontweight='bold')

for ax, (col, label) in zip(axes, present_out.items()):
    if col == 'num_icu_stays':
        vals = df.groupby('analysis_group')[col].mean().reindex(GROUP_ORDER)
        ax.bar(range(4), vals, color=palette, alpha=0.85, edgecolor='white')
        ax.set_ylabel('Mean count')
    else:
        vals = df.groupby('analysis_group')[col].mean().reindex(GROUP_ORDER) * 100
        ax.bar(range(4), vals, color=palette, alpha=0.85, edgecolor='white')
        ax.set_ylabel('%')
    ax.set_xticks(range(4)); ax.set_xticklabels([g.replace(' - ','\n') for g in GROUP_ORDER], fontsize=9)
    ax.set_title(label, fontweight='bold'); ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(vals):
        ax.text(i, v + vals.max()*0.01, f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_outcomes.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: eda_outcomes.png")

print("\n✓ Extended EDA complete.")


# STEP 3.1: BUILD STATIC FEATURE MATRIX
# Excludes: los_days (target leakage), hospital_expire_flag (outcome),
#           discharge_location / DRG codes (post-admission)

In [ ]:

print("=" * 70)
print("STATIC FEATURE MATRIX")
print("=" * 70)

def _dummies(series, prefix, top_n=None):
    s = series.fillna('UNKNOWN').astype(str)
    if top_n:
        top = s.value_counts().head(top_n).index
        s = s.where(s.isin(top), 'OTHER')
    return pd.get_dummies(s, prefix=prefix, drop_first=False)

cohort_reset = cohort_df.reset_index(drop=True)

X_static = pd.DataFrame()

# ── Demographics & admission context ─────────────────────────────────────────
X_static['age_at_icu']            = cohort_reset['age_at_icu'].clip(18, 120)
X_static['gender_male']           = (cohort_reset['gender'] == 'M').astype(int)
X_static['admit_hour']            = cohort_reset['admit_hour']
X_static['admit_dow']             = cohort_reset['admit_dow']
X_static['admit_season']          = cohort_reset['admit_season']
X_static['weekend_admit']         = cohort_reset['weekend_admit']
X_static['ed_los_hours']          = cohort_reset['ed_los_hours'].clip(0, 300)
X_static['from_ed']               = cohort_reset['ed_los_hours'].notna().astype(int)
X_static['icu_admit_delay_hours'] = cohort_reset['icu_admit_delay_hours'].clip(-10, 500)
X_static['num_diagnoses']         = cohort_reset['num_diagnoses'].fillna(0)
X_static['has_fragmentation']     = cohort_reset['has_fragmentation'].fillna(0)
X_static['num_icu_stays']         = cohort_reset['num_icu_stays'].fillna(1)

# Categorical dummies
X_static = pd.concat([X_static,
    _dummies(cohort_reset['admission_type'],     'adm_type'),
    _dummies(cohort_reset['admission_location'],  'adm_loc', top_n=8),
    _dummies(cohort_reset['first_careunit'],      'careunit'),
    _dummies(cohort_reset['insurance'],           'insurance'),
    _dummies(cohort_reset['race'].apply(
        lambda x: x if str(x) in ['WHITE','BLACK/AFRICAN AMERICAN',
                                   'HISPANIC/LATINO','ASIAN'] else 'OTHER'),
             'race'),
], axis=1)

# ── Comorbidities (Charlson) ──────────────────────────────────────────────────
charlson_cols = ['charlson_comorbidity_index','myocardial_infarct',
    'congestive_heart_failure','chronic_pulmonary_disease','renal_disease',
    'severe_liver_disease','metastatic_solid_tumor','diabetes_with_cc',
    'dementia','cerebrovascular_disease']
for col in charlson_cols:
    if col in cohort_reset.columns:
        X_static[col] = cohort_reset[col].fillna(0)

# ── Anthropometrics ───────────────────────────────────────────────────────────
for col, lo, hi in [('height',100,250),('weight',20,300),('bmi',10,80)]:
    if col in cohort_reset.columns:
        X_static[col] = cohort_reset[col].clip(lo, hi)

# ── Severity scores ───────────────────────────────────────────────────────────
for col in ['sapsii_score','apsiii_score','oasis_score','lods_score','first_day_sofa']:
    if col in cohort_reset.columns:
        X_static[col] = cohort_reset[col]

# ── Clinical flags ────────────────────────────────────────────────────────────
for col in ['sepsis3_flag',
            'aki_stage_creat_max','aki_stage_uo_max','aki_max_stage']:
    if col in cohort_reset.columns:
        X_static[col] = cohort_reset[col].fillna(0)

# ── Service context ───────────────────────────────────────────────────────────
if 'admitting_service' in cohort_reset.columns:
    X_static = pd.concat([X_static,
        _dummies(cohort_reset['admitting_service'], 'service', top_n=10)
    ], axis=1)
    X_static['surgical_admission']  = cohort_reset.get('surgical_admission',  pd.Series(0, index=cohort_reset.index)).fillna(0)
    X_static['n_service_transfers'] = cohort_reset.get('n_service_transfers', pd.Series(0, index=cohort_reset.index)).fillna(0)

assert 'los_days' not in X_static.columns, "leakage: los_days must not be in X_static"

print(f"X_static: {X_static.shape[0]:,} stays x {X_static.shape[1]} features")
missing_pct = (X_static.isna().mean() * 100).sort_values(ascending=False)
high_miss = missing_pct[missing_pct > 30]
if len(high_miss):
    print(f"  High-missingness features (>30%):"); print(high_miss.to_string())


# STEP 3.2: TEMPORAL FEATURE MATRIX + DELTA FEATURES

In [ ]:

print("=" * 70)
print("TEMPORAL FEATURE MATRIX (3 x 24h bins + delta features)")
print("=" * 70)

anchor = cohort_reset[['stay_id']].copy()

def pivot_bins(raw_df, value_cols):
    """Long (stay_id, bin, cols) → wide (stay_id, col_b0, col_b1, col_b2)."""
    if raw_df is None or raw_df.empty or 'stay_id' not in raw_df.columns:
        return anchor.copy()
    frames = []
    for b in [0, 1, 2]:
        sub = raw_df[raw_df['bin'] == b][['stay_id'] + value_cols].copy()
        sub = sub.rename(columns={c: f'{c}_b{b}' for c in value_cols})
        frames.append(sub.set_index('stay_id'))
    wide = pd.concat(frames, axis=1).reset_index()
    return anchor.merge(wide, on='stay_id', how='left').drop(columns=['stay_id'])

def add_deltas(df, cols_b0_base):
    """Add Δ(b1-b0) and Δ(b2-b1) for named base columns."""
    for col in cols_b0_base:
        c0, c1, c2 = f'{col}_b0', f'{col}_b1', f'{col}_b2'
        if c0 in df.columns and c1 in df.columns:
            df[f'd_{col}_b1b0'] = df[c1] - df[c0]
        if c1 in df.columns and c2 in df.columns:
            df[f'd_{col}_b2b1'] = df[c2] - df[c1]
    return df

parts = []

# Vitals
vcols = ['hr_mean','hr_min','sbp_mean','sbp_min','mbp_mean','mbp_min',
          'rr_mean','rr_max','temp_mean','spo2_mean','spo2_min']
v = pivot_bins(temporal_frames.get('vitals'), vcols)
v = add_deltas(v, ['hr_mean','sbp_mean','mbp_mean','rr_mean','spo2_mean'])
parts.append(v); print(f"vitals:    {v.shape[1]} cols")

# GCS
g = pivot_bins(temporal_frames.get('gcs'), ['gcs_mean','gcs_min'])
g = add_deltas(g, ['gcs_mean'])
parts.append(g); print(f"gcs:       {g.shape[1]} cols")

# Chemistry
ccols = ['sodium_mean','k_mean','bicarb_mean','bicarb_min',
          'bun_mean','bun_max','cr_mean','cr_max',
          'glucose_mean','albumin_mean','ag_mean','ag_max','ca_mean']
c = pivot_bins(temporal_frames.get('chemistry'), ccols)
c = add_deltas(c, ['cr_mean','bun_mean','k_mean','bicarb_mean','albumin_mean'])
parts.append(c); print(f"chemistry: {c.shape[1]} cols")

# CBC + differential
cbcols = ['hgb_mean','hgb_min','plt_mean','plt_min',
           'wbc_mean','wbc_max','neut_pct_mean','lymph_pct_mean','nlr_mean']
cb = pivot_bins(temporal_frames.get('cbc'), cbcols)
cb = add_deltas(cb, ['hgb_mean','plt_mean','wbc_mean','nlr_mean'])
parts.append(cb); print(f"cbc+diff:  {cb.shape[1]} cols")

# Coagulation
qcols = ['inr_mean','inr_max','ptt_mean','ptt_max','fibrinogen_mean']
q = pivot_bins(temporal_frames.get('coag'), qcols)
q = add_deltas(q, ['inr_mean'])
parts.append(q); print(f"coag:      {q.shape[1]} cols")

# Blood gas
bgcols = ['ph_mean','ph_min','lactate_mean','lactate_max',
           'pf_ratio_mean','pf_ratio_min','be_mean']
bg = pivot_bins(temporal_frames.get('bg'), bgcols)
bg = add_deltas(bg, ['lactate_mean','ph_mean','pf_ratio_mean'])
parts.append(bg); print(f"blood gas: {bg.shape[1]} cols")

# Cardiac markers
cd = pivot_bins(temporal_frames.get('cardiac'),
                ['trop_t_max','ckmb_max','ntprobnp_max'])
parts.append(cd); print(f"cardiac:   {cd.shape[1]} cols")

# Enzymes
ez = pivot_bins(temporal_frames.get('enzymes'),
                ['alt_mean','ast_mean','bili_mean','bili_max'])
ez = add_deltas(ez, ['bili_mean','ast_mean'])
parts.append(ez); print(f"enzymes:   {ez.shape[1]} cols")

# SOFA per bin + slope (primary predictor)
sf = pivot_bins(temporal_frames.get('sofa'), ['sofa_mean','sofa_max'])
sf = add_deltas(sf, ['sofa_mean','sofa_max'])
parts.append(sf); print(f"sofa:      {sf.shape[1]} cols")

# Ventilation + RRT
vt = pivot_bins(temporal_frames.get('vent'),  ['vent_invasive','vent_noninvasive'])
rt = pivot_bins(temporal_frames.get('rrt'),   ['rrt_flag'])
parts.append(vt); parts.append(rt)
print(f"vent+rrt:  {vt.shape[1] + rt.shape[1]} cols")

# Assemble X_temporal
X_temporal = pd.concat(parts, axis=1)
print(f"\nX_temporal: {X_temporal.shape[0]:,} x {X_temporal.shape[1]} features")
print(f"  Delta features: {sum(1 for c in X_temporal.columns if c.startswith('d_'))}")

# ── Combine static + temporal ──────────────────────────────────────────────────
X_combined = pd.concat([
    X_static.reset_index(drop=True),
    X_temporal.reset_index(drop=True)
], axis=1)

print(f"\nX_combined: {X_combined.shape[0]:,} stays x {X_combined.shape[1]} total features")
print(f"  Static:   {X_static.shape[1]}")
print(f"  Temporal: {X_temporal.shape[1]}")


# STEP 3.3: FEATURE PREPROCESSING

In [ ]:

print("\\nPreprocessing features...")

X_combined = X_combined.fillna(X_combined.mean())

variance = X_combined.var()
zero_var_cols = variance[variance == 0].index.tolist()
if len(zero_var_cols) > 0:
    X_combined = X_combined.drop(columns=zero_var_cols)
    print(f"Removed {len(zero_var_cols)} zero-variance features")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)
X_scaled = pd.DataFrame(X_scaled, columns=X_combined.columns)

print(f"✓ Scaling applied (StandardScaler)")

# STEP 3.4: PREPARE TARGET & SPLIT — 3-way: holdout / train / validation

In [ ]:

print("\nPreparing splits...\n")

y = cohort_reset['icu_stay_21plus'].values

print(f"Target Distribution:")
print(f"  <21 days:  {(y==0).sum():,} ({(y==0).mean():.1%})")
print(f"  >=21 days: {(y==1).sum():,} ({(y==1).mean():.1%})")
print(f"  Imbalance: {(y==0).sum() / (y==1).sum():.2f}:1")

# ── Step 1: carve out 20% final holdout (never touched during development) ───
X_work, X_holdout, y_work, y_holdout = train_test_split(
    X_combined, y, test_size=0.20, random_state=NP_SEED, stratify=y
)
X_work_scaled, X_holdout_scaled, _, _ = train_test_split(
    X_scaled, y, test_size=0.20, random_state=NP_SEED, stratify=y
)

# ── Step 2: split working set 80/20 → train / validation ─────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_work, y_work, test_size=0.20, random_state=NP_SEED, stratify=y_work
)
X_train_scaled, X_test_scaled, _, _ = train_test_split(
    X_work_scaled, y_work, test_size=0.20, random_state=NP_SEED, stratify=y_work
)

n_total = len(y)
print(f"\nSplit summary (N={n_total:,}):")
print(f"  Train      : {len(y_train):,}  ({len(y_train)/n_total:.0%} of total)")
print(f"  Validation : {len(y_test):,}  ({len(y_test)/n_total:.0%} of total)")
print(f"  Holdout    : {len(y_holdout):,}  ({len(y_holdout)/n_total:.0%} of total  ← final eval only)")
print(f"\n  Pos rate — train: {y_train.mean():.2%} | val: {y_test.mean():.2%} | holdout: {y_holdout.mean():.2%}")


# STEP 4.1: DEFINE EVALUATION METRICS

In [ ]:

print("\\n" + "="*80)
print("MODEL TRAINING & EVALUATION (AUPRC Focus)")
print("="*80 + "\\n")

def calculate_auprc(y_true, y_pred_proba):
    precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
    return auc(recall, precision)

def calculate_auroc(y_true, y_pred_proba):
    return roc_auc_score(y_true, y_pred_proba)

def evaluate_model(y_true, y_pred_proba, y_pred_binary, model_name='Model'):
    y_pred_proba = np.clip(y_pred_proba, 0.0, 1.0)  # Lasso/Ridge can exceed [0,1]
    metrics = {
        'Model': model_name,
        'AUPRC': calculate_auprc(y_true, y_pred_proba),
        'AUROC': calculate_auroc(y_true, y_pred_proba),
        'Brier': brier_score_loss(y_true, y_pred_proba),
        'Accuracy': accuracy_score(y_true, y_pred_binary),
        'Sensitivity': (y_pred_binary[y_true==1] == 1).sum() / (y_true==1).sum(),
        'Specificity': (y_pred_binary[y_true==0] == 0).sum() / (y_true==0).sum(),
    }
    return metrics

print("✓ Metrics defined")

# STEP 4.2: TRAIN LASSO

In [ ]:

print("\\n" + "-"*60)
print("MODEL 1: LASSO REGRESSION")
print("-"*60)

lasso_cv = LassoCV(cv=5, random_state=NP_SEED, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)

print(f"Optimal alpha: {lasso_cv.alpha_:.6f}")

y_pred_proba_lasso = lasso_cv.predict(X_test_scaled)
y_pred_binary_lasso = (y_pred_proba_lasso >= 0.5).astype(int)

metrics_lasso = evaluate_model(y_test, y_pred_proba_lasso, y_pred_binary_lasso, 'Lasso')

print(f"\\nLasso Performance (Test Set):")
for metric, value in metrics_lasso.items():
    if metric != 'Model':
        print(f"  {metric:15s}: {value:.4f}")

lasso_coef = pd.DataFrame({
    'Feature': X_combined.columns,
    'Coefficient': lasso_cv.coef_
})
lasso_coef['Abs_Coef'] = lasso_coef['Coefficient'].abs()
lasso_coef = lasso_coef[lasso_coef['Abs_Coef'] > 0].sort_values('Abs_Coef', ascending=False)

print(f"\\nLasso: {len(lasso_coef)} non-zero coefficients")

# STEP 4.3: TRAIN RIDGE

In [ ]:

print("\\n" + "-"*60)
print("MODEL 2: RIDGE REGRESSION")
print("-"*60)

ridge_cv = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100, 1000], cv=5)
ridge_cv.fit(X_train_scaled, y_train)

print(f"Optimal alpha: {ridge_cv.alpha_:.4f}")

y_pred_proba_ridge = ridge_cv.predict(X_test_scaled)
y_pred_binary_ridge = (y_pred_proba_ridge >= 0.5).astype(int)

metrics_ridge = evaluate_model(y_test, y_pred_proba_ridge, y_pred_binary_ridge, 'Ridge')

print(f"\\nRidge Performance (Test Set):")
for metric, value in metrics_ridge.items():
    if metric != 'Model':
        print(f"  {metric:15s}: {value:.4f}")

ridge_coef = pd.DataFrame({
    'Feature': X_combined.columns,
    'Coefficient': ridge_cv.coef_
})
ridge_coef['Abs_Coef'] = ridge_coef['Coefficient'].abs()
ridge_coef = ridge_coef.sort_values('Abs_Coef', ascending=False)

# STEP 4.4: TRAIN RANDOM FOREST

In [ ]:

print("\n" + "-"*60)
print("MODEL 3: RANDOM FOREST")
print("-"*60)

class_weight = {0: 1.0, 1: (y_train==0).sum() / (y_train==1).sum()}
print(f"Class weights: {class_weight}")

# ── 5-Fold Cross-Validation (training set only) ───────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=NP_SEED)
cv_results_rf = cross_validate(
    RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=10,
        min_samples_leaf=5, class_weight=class_weight,
        random_state=NP_SEED, n_jobs=-1
    ),
    X_train, y_train, cv=cv,
    scoring={'auprc': 'average_precision', 'auroc': 'roc_auc'},
    n_jobs=-1
)
print(f"\n5-Fold CV (training set):")
print(f"  AUPRC: {cv_results_rf['test_auprc'].mean():.4f} ± {cv_results_rf['test_auprc'].std():.4f}")
print(f"  AUROC: {cv_results_rf['test_auroc'].mean():.4f} ± {cv_results_rf['test_auroc'].std():.4f}")

# ── Final fit on full training set → 20% holdout evaluation ──────────────────
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_split=10,
    min_samples_leaf=5, class_weight=class_weight,
    random_state=NP_SEED, n_jobs=-1, verbose=0
)
rf.fit(X_train, y_train)
print(f"✓ Random Forest trained (200 trees)")

y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]
y_pred_binary_rf = rf.predict(X_test)

metrics_rf = evaluate_model(y_test, y_pred_proba_rf, y_pred_binary_rf, 'Random Forest')

print(f"\nRandom Forest Performance (20% Holdout):")
for metric, value in metrics_rf.items():
    if metric != 'Model':
        print(f"  {metric:15s}: {value:.4f}")

rf_importance = pd.DataFrame({
    'Feature': X_combined.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nTop 5 Features:")
for idx, row in rf_importance.head(5).iterrows():
    print(f"  {row['Feature']:30s}: {row['Importance']*100:6.2f}%")


# STEP 4.5: TRAIN XGBOOST

In [ ]:

print("\n" + "-"*60)
print("MODEL 4: XGBOOST")
print("-"*60)

scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# ── 5-Fold Cross-Validation (training set only) ───────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=NP_SEED)
cv_results_xgb = cross_validate(
    XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=NP_SEED, verbosity=0, eval_metric='logloss'
    ),
    X_train, y_train, cv=cv,
    scoring={'auprc': 'average_precision', 'auroc': 'roc_auc'},
    n_jobs=-1
)
print(f"\n5-Fold CV (training set):")
print(f"  AUPRC: {cv_results_xgb['test_auprc'].mean():.4f} ± {cv_results_xgb['test_auprc'].std():.4f}")
print(f"  AUROC: {cv_results_xgb['test_auroc'].mean():.4f} ± {cv_results_xgb['test_auroc'].std():.4f}")

# ── Final fit on full training set → 20% holdout evaluation ──────────────────
xgb = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight, random_state=NP_SEED,
    verbosity=0, eval_metric='logloss'
)
xgb.fit(X_train, y_train)
print(f"✓ XGBoost trained (200 rounds)")

y_pred_proba_xgb = xgb.predict_proba(X_test)[:, 1]
y_pred_binary_xgb = xgb.predict(X_test)

metrics_xgb = evaluate_model(y_test, y_pred_proba_xgb, y_pred_binary_xgb, 'XGBoost')

print(f"\nXGBoost Performance (Validation):")
for metric, value in metrics_xgb.items():
    if metric != 'Model':
        print(f"  {metric:15s}: {value:.4f}")

xgb_importance = pd.DataFrame({
    'Feature': X_combined.columns,
    'Importance': xgb.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nTop 5 Features:")
for idx, row in xgb_importance.head(5).iterrows():
    print(f"  {row['Feature']:30s}: {row['Importance']*100:6.2f}%")


# STEP 4.6: MODEL COMPARISON

In [ ]:

print("\\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

all_metrics = pd.DataFrame([
    metrics_lasso, metrics_ridge, metrics_rf, metrics_xgb
])

all_metrics_sorted = all_metrics.sort_values('AUPRC', ascending=False)

print("\\nModel Performance (sorted by AUPRC):")
print(all_metrics_sorted[['Model', 'AUPRC', 'AUROC', 'Brier', 'Accuracy', 'Sensitivity', 'Specificity']].to_string(index=False))

best_model = all_metrics_sorted.iloc[0]
print(f"\\n" + "="*80)
print(f"BEST MODEL: {best_model['Model']}")
print(f"="*80)
print(f"  AUPRC: {best_model['AUPRC']:.4f} (PRIMARY METRIC)")
print(f"  AUROC: {best_model['AUROC']:.4f}")
print(f"  Accuracy: {best_model['Accuracy']:.4f}")
print(f"  Brier Score: {best_model['Brier']:.4f} (lower is better)")
print(f"  Sensitivity: {best_model['Sensitivity']:.4f}")
print(f"  Specificity: {best_model['Specificity']:.4f}")

all_metrics_sorted.to_csv('model_comparison_results.csv', index=False)
print(f"\\n✓ Results saved: model_comparison_results.csv")

# STEP 4.7: VISUALIZE MODEL COMPARISON

In [ ]:

print("\\nCreating comparison visualizations...\\n")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

models_list = all_metrics_sorted['Model'].values
auprc_values = all_metrics_sorted['AUPRC'].values
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(models_list))]

axes[0, 0].barh(range(len(models_list)), auprc_values, color=colors, alpha=0.8)
axes[0, 0].set_yticks(range(len(models_list)))
axes[0, 0].set_yticklabels(models_list, fontweight='bold')
axes[0, 0].set_xlabel('AUPRC', fontweight='bold')
axes[0, 0].set_title('Model Comparison: AUPRC (Primary)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlim([0, 1])
axes[0, 0].grid(axis='x', alpha=0.3)

for i, (model, val) in enumerate(zip(models_list, auprc_values)):
    axes[0, 0].text(val + 0.02, i, f'{val:.4f}', va='center', fontweight='bold')

axes[0, 1].scatter(all_metrics['AUROC'], all_metrics['AUPRC'],
                    s=400, c=range(len(all_metrics)), cmap='viridis',
                    alpha=0.7, edgecolors='black', linewidth=2)
for idx, row in all_metrics.iterrows():
    axes[0, 1].annotate(row['Model'], (row['AUROC'], row['AUPRC']),
                        xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')
axes[0, 1].set_xlabel('AUROC', fontweight='bold')
axes[0, 1].set_ylabel('AUPRC', fontweight='bold')
axes[0, 1].set_title('AUPRC vs AUROC', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xlim([0.5, 1.0])
axes[0, 1].set_ylim([0.3, 1.0])

axes[1, 0].scatter(all_metrics['Specificity'], all_metrics['Sensitivity'],
                    s=400, c=all_metrics['AUPRC'], cmap='RdYlGn',
                    alpha=0.7, edgecolors='black', linewidth=2, vmin=0.4, vmax=1.0)
for idx, row in all_metrics.iterrows():
    axes[1, 0].annotate(row['Model'], (row['Specificity'], row['Sensitivity']),
                        xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')
axes[1, 0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1, 0].set_xlabel('Specificity', fontweight='bold')
axes[1, 0].set_ylabel('Sensitivity', fontweight='bold')
axes[1, 0].set_title('Sensitivity vs Specificity', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xlim([0, 1])
axes[1, 0].set_ylim([0, 1])

metrics_to_plot = ['AUPRC', 'AUROC', 'Accuracy', 'Sensitivity', 'Specificity']
x_pos = np.arange(len(metrics_to_plot))
width = 0.2

for i, (idx, row) in enumerate(all_metrics_sorted.iterrows()):
    values = [row[metric] for metric in metrics_to_plot]
    axes[1, 1].bar(x_pos + i*width, values, width, label=row['Model'], alpha=0.8)

axes[1, 1].set_ylabel('Score', fontweight='bold')
axes[1, 1].set_title('Comprehensive Metrics', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(x_pos + width*1.5)
axes[1, 1].set_xticklabels(metrics_to_plot, rotation=45, ha='right')
axes[1, 1].legend(loc='lower left', fontsize=9)
axes[1, 1].grid(axis='y', alpha=0.3)
axes[1, 1].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('05_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 05_model_comparison.png")

# STEP 4.7b: CONFUSION MATRICES — VALIDATION SET

In [ ]:

from sklearn.metrics import confusion_matrix

print("\nValidation set confusion matrices (threshold = 0.5)\n")

val_preds = {
    'Lasso':         (y_pred_binary_lasso,  y_test),
    'Ridge':         (y_pred_binary_ridge,  y_test),
    'Random Forest': (y_pred_binary_rf,     y_test),
    'XGBoost':       (y_pred_binary_xgb,    y_test),
}
cm_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (y_pred, y_true)), color in zip(axes, val_preds.items(), cm_colors):
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum() * 100
    ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(name, fontweight='bold', fontsize=12, pad=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Neg','Pos'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['Neg','Pos'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            lbl = ['TN','FP','FN','TP'][i*2+j]
            ax.text(j, i, f'{lbl}\n{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)',
                    ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white' if cm[i,j] > thresh else 'black')
plt.suptitle('Confusion Matrices — Validation Set', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices_val.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: confusion_matrices_val.png")


# STEP 4.7c: CALIBRATION PLOTS — VALIDATION SET

In [ ]:

from sklearn.calibration import calibration_curve

print("\nCalibration plots (validation set)\n")

val_probas = {
    'Lasso':         np.clip(y_pred_proba_lasso, 0.0, 1.0),
    'Ridge':         np.clip(y_pred_proba_ridge, 0.0, 1.0),
    'Random Forest': y_pred_proba_rf,
    'XGBoost':       y_pred_proba_xgb,
}

cal_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, (name, proba), color in zip(axes[:4], val_probas.items(), cal_colors):
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy='quantile')
    brier = brier_score_loss(y_test, proba)
    ax.plot([0,1],[0,1], 'k--', lw=1.5, label='Perfect')
    ax.plot(mean_pred, frac_pos, 'o-', color=color, lw=2, ms=6, label=f'Brier={brier:.4f}')
    ax.fill_between(mean_pred, frac_pos, mean_pred, alpha=0.12, color=color)
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlabel('Mean predicted prob'); ax.set_ylabel('Fraction positives')
    ax.set_xlim([0,1]); ax.set_ylim([0,1])
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax_hist = ax.inset_axes([0.54, 0.04, 0.42, 0.28])
    ax_hist.hist(proba[y_test==0], bins=20, color='steelblue', alpha=0.6, density=True, label='Neg')
    ax_hist.hist(proba[y_test==1], bins=20, color='salmon',    alpha=0.7, density=True, label='Pos')
    ax_hist.set_xlabel('p̂', fontsize=7); ax_hist.set_yticks([])
    ax_hist.legend(fontsize=6); ax_hist.tick_params(labelsize=6)

# Overlay panel
ax_all = axes[4]
ax_all.plot([0,1],[0,1], 'k--', lw=1.5, label='Perfect')
for (name, proba), color in zip(val_probas.items(), cal_colors):
    fp, mp = calibration_curve(y_test, proba, n_bins=10, strategy='quantile')
    b = brier_score_loss(y_test, proba)
    ax_all.plot(mp, fp, 'o-', color=color, lw=2, ms=4, label=f'{name} (B={b:.4f})')
ax_all.set_title('All Models', fontweight='bold', fontsize=11)
ax_all.set_xlabel('Mean predicted prob'); ax_all.set_ylabel('Fraction positives')
ax_all.set_xlim([0,1]); ax_all.set_ylim([0,1])
ax_all.legend(fontsize=7, loc='upper left'); ax_all.grid(True, alpha=0.3)

plt.suptitle('Calibration Plots — Validation Set', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('calibration_plots_val.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: calibration_plots_val.png")


# STEP 4.8: PRECISION-RECALL & ROC CURVES

In [ ]:

print("\\nGenerating PR and ROC curves...\\n")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

predictions = {
    'Lasso': y_pred_proba_lasso,
    'Ridge': y_pred_proba_ridge,
    'RF': y_pred_proba_rf,
    'XGBoost': y_pred_proba_xgb
}

colors_models = plt.cm.Set1(np.linspace(0, 1, len(predictions)))

for (model_name, y_pred), color in zip(predictions.items(), colors_models):
    precision, recall, _ = precision_recall_curve(y_test, y_pred)
    auprc_score = auc(recall, precision)
    axes[0].plot(recall, precision, color=color, linewidth=2.5,
                 label=f'{model_name} (AUPRC={auprc_score:.4f})',
                 marker='o', markevery=max(1, len(recall)//10))

baseline_precision = (y_test == 1).mean()
axes[0].axhline(y=baseline_precision, color='gray', linestyle='--',
                 linewidth=2, label=f'Baseline={baseline_precision:.4f}')
axes[0].set_xlabel('Recall (Sensitivity)', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Precision', fontweight='bold', fontsize=11)
axes[0].set_title('Precision-Recall Curves', fontsize=13, fontweight='bold')
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

for (model_name, y_pred), color in zip(predictions.items(), colors_models):
    fpr, tpr, _ = roc_curve(y_test, y_pred)
    auroc_score = roc_auc_score(y_test, y_pred)
    axes[1].plot(fpr, tpr, color=color, linewidth=2.5,
                 label=f'{model_name} (AUROC={auroc_score:.4f})',
                 marker='o', markevery=max(1, len(fpr)//10))

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random (AUROC=0.50)')
axes[1].set_xlabel('False Positive Rate', fontweight='bold', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontweight='bold', fontsize=11)
axes[1].set_title('ROC Curves', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])
axes[1].set_aspect('equal')

plt.tight_layout()
plt.savefig('06_pr_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 06_pr_roc_curves.png")

# STEP 4.9: FEATURE IMPORTANCE COMPARISON

In [ ]:

print("\\nComparing feature importance...\\n")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

rf_top15 = rf_importance.head(15).sort_values('Importance')
axes[0].barh(range(len(rf_top15)), rf_top15['Importance']*100, color='steelblue', alpha=0.8)
axes[0].set_yticks(range(len(rf_top15)))
axes[0].set_yticklabels(rf_top15['Feature'], fontsize=10)
axes[0].set_xlabel('Importance (%)', fontweight='bold')
axes[0].set_title('Random Forest: Top 15 Features', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

xgb_top15 = xgb_importance.head(15).sort_values('Importance')
axes[1].barh(range(len(xgb_top15)), xgb_top15['Importance']*100, color='coral', alpha=0.8)
axes[1].set_yticks(range(len(xgb_top15)))
axes[1].set_yticklabels(xgb_top15['Feature'], fontsize=10)
axes[1].set_xlabel('Importance (%)', fontweight='bold')
axes[1].set_title('XGBoost: Top 15 Features', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('07_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 07_feature_importance.png")

# STEP 5.1: FINAL SUMMARY

In [ ]:

print("\\n" + "="*80)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("="*80)

print(f"\\n1. COHORT CHARACTERISTICS")
print(f"   Overall (N={len(cohort_df)}):")
print(f"     Age: {cohort_df['age_at_icu'].mean():.1f} +/- {cohort_df['age_at_icu'].std():.1f} years")
print(f"     Mortality: {cohort_df['hospital_expire_flag'].mean():.1%}")

print(f"\\n   21+ Days: {(cohort_df['icu_stay_21plus']==1).sum()} patients")
subset_21 = cohort_df[cohort_df['icu_stay_21plus']==1]
print(f"     Mortality: {subset_21['hospital_expire_flag'].mean():.1%}")

print(f"\\n   <21 Days: {(cohort_df['icu_stay_21plus']==0).sum()} patients")
subset_less = cohort_df[cohort_df['icu_stay_21plus']==0]
print(f"     Mortality: {subset_less['hospital_expire_flag'].mean():.1%}")

print(f"\\n2. BEST MODEL: {best_model['Model']}")
print(f"   AUPRC: {best_model['AUPRC']:.4f}")
print(f"   AUROC: {best_model['AUROC']:.4f}")
print(f"   Sensitivity: {best_model['Sensitivity']:.1%}")
print(f"   Specificity: {best_model['Specificity']:.1%}")

print(f"\\nANALYSIS COMPLETE")
print(f"Generated: 7 figures + model_comparison_results.csv")

# STEP 5.2: FINAL HOLDOUT EVALUATION
# All 4 models evaluated on the 20% holdout — data they have never seen.

In [ ]:

print("\n" + "="*80)
print("FINAL HOLDOUT EVALUATION (20% — never seen during training or validation)")
print("="*80)

holdout_results = []

# ── Lasso ─────────────────────────────────────────────────────────────────────
y_h_proba_lasso  = lasso_cv.predict(X_holdout_scaled)
y_h_binary_lasso = (y_h_proba_lasso >= 0.5).astype(int)
holdout_results.append(evaluate_model(y_holdout, y_h_proba_lasso, y_h_binary_lasso, 'Lasso'))

# ── Ridge ─────────────────────────────────────────────────────────────────────
y_h_proba_ridge  = ridge_cv.predict(X_holdout_scaled)
y_h_binary_ridge = (y_h_proba_ridge >= 0.5).astype(int)
holdout_results.append(evaluate_model(y_holdout, y_h_proba_ridge, y_h_binary_ridge, 'Ridge'))

# ── Random Forest ─────────────────────────────────────────────────────────────
y_h_proba_rf  = rf.predict_proba(X_holdout)[:, 1]
y_h_binary_rf = rf.predict(X_holdout)
holdout_results.append(evaluate_model(y_holdout, y_h_proba_rf, y_h_binary_rf, 'Random Forest'))

# ── XGBoost ───────────────────────────────────────────────────────────────────
y_h_proba_xgb  = xgb.predict_proba(X_holdout)[:, 1]
y_h_binary_xgb = xgb.predict(X_holdout)
holdout_results.append(evaluate_model(y_holdout, y_h_proba_xgb, y_h_binary_xgb, 'XGBoost'))


# ── Summary table ─────────────────────────────────────────────────────────────
holdout_df = pd.DataFrame(holdout_results).sort_values('AUPRC', ascending=False)
print("\nHoldout Performance (sorted by AUPRC):")
print(holdout_df[['Model','AUPRC','AUROC','Brier','Accuracy','Sensitivity','Specificity']].to_string(index=False))

best_h = holdout_df.iloc[0]
print(f"\n{'='*80}")
print(f"BEST MODEL ON HOLDOUT: {best_h['Model']}")
print(f"  AUPRC : {best_h['AUPRC']:.4f}")
print(f"  AUROC : {best_h['AUROC']:.4f}")
print(f"  Brier : {best_h['Brier']:.4f}")
print(f"  Sens  : {best_h['Sensitivity']:.4f}  |  Spec: {best_h['Specificity']:.4f}")
print(f"{'='*80}")

# ── PR + ROC curves on holdout ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

holdout_preds = {
    'Lasso':         y_h_proba_lasso,
    'Ridge':         y_h_proba_ridge,
    'Random Forest': y_h_proba_rf,
    'XGBoost':       y_h_proba_xgb,
}
colors = ['#3498db','#e74c3c','#2ecc71','#f39c12']

for ax, (title, curve_fn) in zip(axes, [
    ('Precision-Recall — Holdout', 'pr'),
    ('ROC — Holdout',              'roc'),
]):
    for (name, proba), color in zip(holdout_preds.items(), colors):
        if curve_fn == 'pr':
            prec, rec, _ = precision_recall_curve(y_holdout, proba)
            score = auc(rec, prec)
            ax.plot(rec, prec, color=color, lw=2, label=f'{name} (AUPRC={score:.3f})')
            ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
            baseline = y_holdout.mean()
            ax.axhline(baseline, ls='--', color='grey', lw=1, label=f'Baseline ({baseline:.3f})')
        else:
            fpr, tpr, _ = roc_curve(y_holdout, proba)
            score = roc_auc_score(y_holdout, proba)
            ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUROC={score:.3f})')
            ax.plot([0,1],[0,1], 'k--', lw=1)
            ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/holdout_curves.png', dpi=150)
plt.show()
print("✓ Holdout curves saved.")

# ── Confusion matrices — Holdout ─────────────────────────────────────────────
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve

print("\nHoldout confusion matrices\n")
hold_binary_map = {
    'Lasso':         (y_h_binary_lasso,  y_holdout),
    'Ridge':         (y_h_binary_ridge,  y_holdout),
    'Random Forest': (y_h_binary_rf,     y_holdout),
    'XGBoost':       (y_h_binary_xgb,    y_holdout),
}
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (yp, yt)) in zip(axes, hold_binary_map.items()):
    cm = confusion_matrix(yt, yp)
    cm_pct = cm.astype(float) / cm.sum() * 100
    ax.imshow(cm, interpolation='nearest', cmap='Greens')
    ax.set_title(name, fontweight='bold', fontsize=12, pad=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Neg','Pos'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['Neg','Pos'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            lbl = ['TN','FP','FN','TP'][i*2+j]
            ax.text(j, i, f'{lbl}\n{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)',
                    ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white' if cm[i,j] > thresh else 'black')
plt.suptitle('Confusion Matrices — Holdout Set', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrices_holdout.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: confusion_matrices_holdout.png")

# ── Calibration plots — Holdout ───────────────────────────────────────────────
print("\nHoldout calibration plots\n")
hold_probas = {
    'Lasso':         np.clip(y_h_proba_lasso, 0.0, 1.0),
    'Ridge':         np.clip(y_h_proba_ridge, 0.0, 1.0),
    'Random Forest': y_h_proba_rf,
    'XGBoost':       y_h_proba_xgb,
}

cal_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, proba), color in zip(axes[:4], hold_probas.items(), cal_colors):
    frac_pos, mean_pred = calibration_curve(y_holdout, proba, n_bins=10, strategy='quantile')
    brier = brier_score_loss(y_holdout, proba)
    ax.plot([0,1],[0,1], 'k--', lw=1.5, label='Perfect')
    ax.plot(mean_pred, frac_pos, 'o-', color=color, lw=2, ms=6, label=f'Brier={brier:.4f}')
    ax.fill_between(mean_pred, frac_pos, mean_pred, alpha=0.12, color=color)
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlabel('Mean predicted prob'); ax.set_ylabel('Fraction positives')
    ax.set_xlim([0,1]); ax.set_ylim([0,1])
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax_all = axes[4]
ax_all.plot([0,1],[0,1], 'k--', lw=1.5, label='Perfect')
for (name, proba), color in zip(hold_probas.items(), cal_colors):
    fp, mp = calibration_curve(y_holdout, proba, n_bins=10, strategy='quantile')
    b = brier_score_loss(y_holdout, proba)
    ax_all.plot(mp, fp, 'o-', color=color, lw=2, ms=4, label=f'{name} (B={b:.4f})')
ax_all.set_title('All Models', fontweight='bold', fontsize=11)
ax_all.set_xlabel('Mean predicted prob'); ax_all.set_ylabel('Fraction positives')
ax_all.set_xlim([0,1]); ax_all.set_ylim([0,1])
ax_all.legend(fontsize=7, loc='upper left'); ax_all.grid(True, alpha=0.3)
plt.suptitle('Calibration Plots — Holdout Set', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/calibration_plots_holdout.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: calibration_plots_holdout.png")


# STEP 5.3: SAVE & DOWNLOAD ALL OUTPUTS

In [ ]:

import joblib, os, zipfile, shutil
from google.colab import files

SAVE_DIR = '/content/icu_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving models and results...")

# Models
joblib.dump(lasso_cv, f'{SAVE_DIR}/model_lasso.pkl');  print("  ✓ Lasso")
joblib.dump(ridge_cv, f'{SAVE_DIR}/model_ridge.pkl');  print("  ✓ Ridge")
joblib.dump(rf,       f'{SAVE_DIR}/model_rf.pkl');     print("  ✓ Random Forest")
xgb.save_model(       f'{SAVE_DIR}/model_xgb.json');   print("  ✓ XGBoost")

# CSVs
cohort_df.to_csv(         f'{SAVE_DIR}/cohort_df.csv',          index=False)
all_metrics_sorted.to_csv(f'{SAVE_DIR}/validation_metrics.csv', index=False)
holdout_df.to_csv(        f'{SAVE_DIR}/holdout_metrics.csv',    index=False)
print("  ✓ CSVs (cohort, validation metrics, holdout metrics)")

# Copy validation plots generated earlier
for png in ['confusion_matrices_val.png', 'calibration_plots_val.png',
            '05_model_comparison.png', '06_pr_roc_curves.png',
            '07_feature_importance.png']:
    if os.path.exists(f'/content/{png}'):
        shutil.copy(f'/content/{png}', f'{SAVE_DIR}/{png}')

# Zip
ZIP_PATH = '/content/icu_outputs.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(SAVE_DIR):
        zf.write(os.path.join(SAVE_DIR, fname), fname)

zip_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"\n✓ Zipped → icu_outputs.zip ({zip_mb:.1f} MB)")
print("  Contents:")
for fname in sorted(os.listdir(SAVE_DIR)):
    sz = os.path.getsize(os.path.join(SAVE_DIR, fname)) / 1e6
    print(f"    {fname:<42s} {sz:5.1f} MB")

files.download(ZIP_PATH)
print("\n✓ Download triggered.")


---

## Notebook Documentation

### Setup & Authentication
- Google Colab Enterprise authentication with service account
- MIMIC-IV 3.1 on BigQuery + local DuckDB (`mimic_iv_derived.duckdb`) on Google Drive
- BigQuery used for: cohort, EDA tables (diagnoses, procedures, services)
- DuckDB used for: all derived features (scores, vitals, labs, temporal features)

---

### Section 0 — Setup
| Step | Cell | Description |
|------|------|-------------|
| 0.0 | 1 | Google Colab authentication & project config |
| 0.1 | 2 | Import libraries |
| 0.1b | 3 | Mount Google Drive, connect DuckDB |
| 0.2 | 4 | Initialize BigQuery client |

---

### Section 1 — Data Extraction
| Step | Cell | Description |
|------|------|-------------|
| 1.0 | 5 | Cohort extraction from BigQuery (adults ≥18, hospital stay ≥72h, first ICU stay only) |
| 1.1 | 6 | Create binary target: ICU LOS ≥21 days = 1 |
| 1.2 | 7 | Static derived features via DuckDB (Charlson, SAPSII, APSIII, OASIS, LODS, SOFA, anthropometrics, sepsis3, AKI) |
| 1.3 | 8 | Temporal features via DuckDB — 3 × 24h bins (0–24h, 24–48h, 48–72h): vitals, GCS, labs, coagulation, blood gas, cardiac markers, enzymes, ventilation, RRT |
| 1.4 | 9 | Merge all features onto cohort_df; procedures query via BigQuery (hadm_id join); export cohort snapshot CSV to Drive |

---

### Section 2 — EDA
| Step | Cell | Description |
|------|------|-------------|
| 2.1 | 10 | Create 4 stratified cohorts: 21+/Died, 21+/Survived, <21/Died, <21/Survived |
| 2.2 | 11 | Insurance type & mortality analysis — 4-group comparison |
| 2.3 | 12 | Diagnosis & LOS analysis — top 15 diagnoses by volume (joins via icustays on hadm_id) |
| 2.4 | 13 | Procedure utilization — 4-group comparison |
| 2.5 | 14 | Readmission & ICU fragmentation analysis |

---

### Section 3 — Feature Engineering
| Step | Cell | Description |
|------|------|-------------|
| 3.1 | 15 | Static feature matrix (demographics, admission type, comorbidities, severity scores) |
| 3.2 | 16 | Temporal feature matrix + delta features (bin-to-bin changes) |
| 3.3 | 17 | StandardScaler preprocessing; combined feature matrix |
| 3.4 | 18 | 3-way stratified split: 20% final holdout / 64% train / 16% validation |

---

### Section 4 — Predictive Modeling
| Step | Cell | Description |
|------|------|-------------|
| 4.1 | 19 | Define evaluation metrics: AUPRC (primary), AUROC, Brier score, Accuracy, Sensitivity, Specificity |
| 4.2 | 20 | Lasso — 5-fold CV alpha selection (LassoCV) |
| 4.3 | 21 | Ridge — 5-fold CV alpha selection (RidgeCV) |
| 4.4 | 22 | Random Forest — 5-fold CV + final fit (200 trees, class_weight balanced) |
| 4.5 | 23 | XGBoost — 5-fold CV + final fit (200 rounds, scale_pos_weight) |
| 4.6 | 24 | Model comparison table (AUPRC, AUROC, Brier, Accuracy, Sensitivity, Specificity) |
| 4.7 | 25 | 4-panel model comparison visualization |
| 4.7b | 26 | Confusion matrices — validation set (all 4 models) |
| 4.7c | 27 | Calibration plots — validation set (reliability curves + Brier + probability histograms) |
| 4.8 | 28 | Precision-Recall and ROC curves — validation set |
| 4.9 | 29 | Feature importance comparison — RF vs XGBoost (top 15) |

---

### Section 5 — Summary & Export
| Step | Cell | Description |
|------|------|-------------|
| 5.1 | 30 | Final cohort summary and best model report |
| 5.2 | 31 | Holdout evaluation (20% never-seen data): all 4 models, PR/ROC curves, confusion matrices, calibration plots |
| 5.3 | 32 | Save & download: models (pkl/json) + CSVs + plots → `icu_outputs.zip` |

---

### Key Design Decisions
- **DuckDB for derived features** — avoids BigQuery quota limits and chunked-query failures
- **3-way split** — 20% holdout is never touched until final evaluation
- **5-fold stratified CV** — on training set only for RF and XGBoost; built into LassoCV/RidgeCV
- **AUPRC as primary metric** — appropriate for imbalanced classes (~2.3% positive)
- **Brier score** — measures probability calibration quality (lower = better)
- **Lasso/Ridge probabilities clipped to [0,1]** — linear models can predict outside this range

---

### Expected Outputs

**Figures** (saved locally + in Drive):
- `05_model_comparison.png`
- `confusion_matrices_val.png`
- `calibration_plots_val.png`
- `06_pr_roc_curves.png`
- `07_feature_importance.png`
- `holdout_curves.png` (Drive)
- `confusion_matrices_holdout.png` (Drive)
- `calibration_plots_holdout.png` (Drive)

**Data files** (in `icu_outputs.zip`):
- `model_lasso.pkl`, `model_ridge.pkl`, `model_rf.pkl` — sklearn models
- `model_xgb.json` — XGBoost native format
- `cohort_df.csv` — full merged feature dataframe
- `validation_metrics.csv`, `holdout_metrics.csv` — results tables

---
